In [ ]:
CONFIG = {
    "input_root": "/content/input_data",
    "output_root": "/content/chat_exports",
    "subset": "user",  # Options: "both", "user", "assistant"
    "format": "md",     # Options: "md", "txt", "json"
    "export_code": True,
    "keywords": ["tau","3:4:5","universe","incompleteness","choice","distinguish","functional role","geometry","algebra","zeta","dimensionless","observer","observation","time","temperature"],     # e.g., ["python", "javascript", "api"]
    "groups": "non-unique",  # Options: "non-unique", "unique"
    "drive_out": None,  # e.g., "/content/drive/MyDrive/chat_exports"
    "max_filename_len": 16000,
    "dry_run": False,
    "verbose": True
}

In [ ]:
# %% [markdown]
# # Chat Export Tool for Google Colab
#
# Processes chat exports from ChatGPT, Claude, and Gemini platforms with CLI-like configuration.
#
# **Quick Start:**
# 1. Run all cells in order
# 2. Place your JSON files in `/content/input_data/`
# 3. Configure settings in the Config cell
# 4. Run the Process cell

# %% Install dependencies
!pip install -q ijson 2>/dev/null && echo "✓ ijson installed (for large file support)" || echo "⚠ ijson not installed"

# %% Import libraries and setup
import json
import os
import re
import sys
import hashlib
import gzip
import zipfile
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Union, Tuple, Any
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime
from glob import glob

# Create directories
Path("/content/input_data").mkdir(exist_ok=True)
Path("/content/chat_exports").mkdir(exist_ok=True)

print("✓ Directories created")
print("  Input:  /content/input_data/")
print("  Output: /content/chat_exports/")

# %% [markdown]
# ## Configuration
#
# Modify these settings as needed:

# %% Configuration settings
CONFIG = {
    "input_root": "/content/input_data",
    "output_root": "/content/chat_exports",
    "subset": "both",  # Options: "both", "user", "assistant"
    "format": "md",     # Options: "md", "txt", "json"
    "export_code": True,
    "keywords": ["word"],     # e.g., ["python", "javascript", "api"]
    "groups": "non-unique",  # Options: "non-unique", "unique"
    "drive_out": None,  # e.g., "/content/drive/MyDrive/chat_exports"
    "max_filename_len": 160,
    "dry_run": False,
    "verbose": True
}

# Print current configuration
print("Current Configuration:")
print("-" * 40)
for key, value in CONFIG.items():
    print(f"{key:20s}: {value}")

# %% Data structures
@dataclass
class Message:
    """Represents a single message in a conversation"""
    role: str  # 'user' or 'assistant'
    content: str
    timestamp: Optional[str] = None
    metadata: Dict = field(default_factory=dict)

@dataclass
class Conversation:
    """Represents a complete conversation"""
    id: str
    title: str
    messages: List[Message]
    created_at: Optional[str] = None
    updated_at: Optional[str] = None
    platform: str = "unknown"
    metadata: Dict = field(default_factory=dict)

# %% Platform parsers
class ChatGPTParser:
    """Parser for ChatGPT conversation exports"""

    @staticmethod
    def parse(data: Dict) -> List[Conversation]:
        """Parse ChatGPT conversations.json format"""
        conversations = []

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')
            created_at = conv_data.get('create_time')
            updated_at = conv_data.get('update_time')

            messages = []
            mapping = conv_data.get('mapping', {})

            # Build message tree from mapping
            for node_id, node_data in mapping.items():
                message = node_data.get('message')
                if message and message.get('content'):
                    content_parts = message['content'].get('parts', [])
                    if content_parts:
                        content = '\n'.join(str(part) for part in content_parts)
                        role = message['author']['role']
                        # Map ChatGPT roles to our standard roles
                        if role == 'system':
                            continue  # Skip system messages
                        role = 'user' if role == 'user' else 'assistant'

                        msg = Message(
                            role=role,
                            content=content,
                            timestamp=message.get('create_time'),
                            metadata={'id': message.get('id')}
                        )
                        messages.append(msg)

            if messages:  # Only add if conversation has messages
                conversations.append(Conversation(
                    id=conv_id,
                    title=title,
                    messages=messages,
                    created_at=created_at,
                    updated_at=updated_at,
                    platform='chatgpt'
                ))

        return conversations

class ClaudeParser:
    """Parser for Claude conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Claude export format"""
        conversations = []

        # Handle both single conversation and multiple conversations
        if isinstance(data, dict):
            data = [data]

        for conv_data in data:
            # Try multiple possible formats
            conv_id = conv_data.get('id', conv_data.get('uuid', ''))
            title = conv_data.get('title', conv_data.get('name', 'Untitled'))

            messages = []
            # Check different possible message locations
            message_list = conv_data.get('messages', conv_data.get('chat_messages',
                                         conv_data.get('conversation', [])))

            for msg_data in message_list:
                # Handle different role naming conventions
                role = msg_data.get('role', msg_data.get('sender', ''))
                if role in ['human', 'user']:
                    role = 'user'
                elif role in ['assistant', 'claude', 'ai']:
                    role = 'assistant'
                else:
                    continue  # Skip unknown roles

                # Get content from various possible fields
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle content that might be in nested structure
                if isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)
                elif isinstance(content, dict):
                    content = content.get('text', str(content))

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp', msg_data.get('created_at')),
                        metadata=msg_data.get('metadata', {})
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='claude'
                ))

        return conversations

class GeminiParser:
    """Parser for Gemini conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Gemini export format"""
        conversations = []

        # Handle both single and multiple conversations
        if isinstance(data, dict):
            if 'conversations' in data:
                data = data['conversations']
            else:
                data = [data]

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')

            messages = []
            # Check for messages in different possible locations
            message_list = conv_data.get('messages', conv_data.get('entries',
                                         conv_data.get('chats', [])))

            for msg_data in message_list:
                # Determine role
                role = msg_data.get('role', msg_data.get('type', ''))
                if role in ['user', 'prompt']:
                    role = 'user'
                elif role in ['model', 'gemini', 'response', 'assistant']:
                    role = 'assistant'
                else:
                    continue

                # Extract content
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle nested content structures
                if isinstance(content, dict):
                    content = content.get('text', str(content))
                elif isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp'),
                        metadata=msg_data.get('metadata', {})
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='gemini'
                ))

        return conversations

class UniversalChatParser:
    """Universal parser that auto-detects platform format"""

    @staticmethod
    def detect_platform(data: Union[Dict, List]) -> str:
        """Detect which platform the data is from"""

        # Convert to dict if needed for checking
        sample = data[0] if isinstance(data, list) and data else data

        # ChatGPT indicators
        if isinstance(sample, dict):
            if 'mapping' in sample:
                return 'chatgpt'
            # Claude indicators
            if 'chat_messages' in sample or 'uuid' in sample:
                return 'claude'
            # Gemini indicators
            if 'entries' in sample or ('messages' in sample and
                any('model' in msg for msg in sample.get('messages', []))):
                return 'gemini'

        # Try to detect from message structure
        if 'messages' in sample:
            msgs = sample['messages']
            if msgs and isinstance(msgs[0], dict):
                if 'author' in msgs[0]:
                    return 'chatgpt'
                if 'sender' in msgs[0]:
                    return 'claude'
                if 'model' in msgs[0] or 'type' in msgs[0]:
                    return 'gemini'

        return 'unknown'

    @staticmethod
    def parse(data: Union[Dict, List], platform: Optional[str] = None) -> List[Conversation]:
        """Parse chat data from any supported platform"""

        if platform is None:
            platform = UniversalChatParser.detect_platform(data)

        parsers = {
            'chatgpt': ChatGPTParser(),
            'claude': ClaudeParser(),
            'gemini': GeminiParser()
        }

        parser = parsers.get(platform, ClaudeParser())  # Default to Claude parser
        return parser.parse(data)

# %% Code extraction
class CodeExtractor:
    """Extract code blocks from messages"""

    LANGUAGE_EXTENSIONS = {
        'python': '.py', 'javascript': '.js', 'typescript': '.ts',
        'java': '.java', 'cpp': '.cpp', 'c': '.c', 'csharp': '.cs',
        'html': '.html', 'css': '.css', 'sql': '.sql', 'bash': '.sh',
        'shell': '.sh', 'yaml': '.yml', 'json': '.json', 'xml': '.xml',
        'markdown': '.md', 'rust': '.rs', 'go': '.go', 'ruby': '.rb',
        'php': '.php', 'swift': '.swift', 'kotlin': '.kt', 'r': '.r'
    }

    @staticmethod
    def extract_code_blocks(content: str) -> List[Tuple[str, str]]:
        """Extract code blocks from content
        Returns: List of (code, language) tuples"""

        code_blocks = []

        # Pattern for ```language\ncode\n```
        pattern = r'```(\w+)?\n?(.*?)```'
        matches = re.findall(pattern, content, re.DOTALL)

        for language, code in matches:
            language = language.lower() if language else 'text'
            code_blocks.append((code.strip(), language))

        # Also check for inline code if no blocks found
        if not code_blocks:
            # Look for common code patterns
            inline_pattern = r'`([^`]+)`'
            inline_matches = re.findall(inline_pattern, content)
            for code in inline_matches:
                if len(code) > 50:  # Only consider substantial inline code
                    code_blocks.append((code, 'text'))

        return code_blocks

# %% Export functionality
class ChatExporter:
    """Export conversations in various formats"""

    def __init__(self, output_root: str, max_filename_len: int = 160, verbose: bool = False):
        self.output_root = Path(output_root)
        self.max_filename_len = max_filename_len
        self.verbose = verbose
        self.exported_files = []
        self.filename_cache = {}  # Track used filenames for collision detection

        # Create output directories
        self.default_dir = self.output_root / "default"
        self.code_dir = self.output_root / "code_exports"

        self.default_dir.mkdir(parents=True, exist_ok=True)
        self.code_dir.mkdir(parents=True, exist_ok=True)

    def _sanitize_filename(self, filename: str) -> str:
        """Sanitize filename for filesystem"""
        # Remove/replace invalid characters
        invalid_chars = '<>:"/\\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')
        # Limit length
        return filename[:self.max_filename_len]

    def _generate_unique_filename(self, base_name: str, directory: Path, ext: str) -> Path:
        """Generate unique filename with collision handling"""
        # Check if filename already used
        filepath = directory / f"{base_name}.{ext}"

        if filepath not in self.filename_cache:
            self.filename_cache[filepath] = True
            return filepath

        # Add hash for uniqueness
        for i in range(1, 100):
            hash_suffix = hashlib.md5(f"{base_name}{i}".encode()).hexdigest()[:6]
            unique_name = f"{base_name}-{hash_suffix}"
            filepath = directory / f"{unique_name}.{ext}"

            if filepath not in self.filename_cache:
                self.filename_cache[filepath] = True
                return filepath

        # Fallback with timestamp
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        filepath = directory / f"{base_name}-{timestamp}.{ext}"
        self.filename_cache[filepath] = True
        return filepath

    def export_conversation(self, conversation: Conversation, subset: str, format: str,
                          target_dir: Optional[Path] = None) -> Optional[Path]:
        """Export a single conversation"""

        # Generate filename components
        safe_title = self._sanitize_filename(conversation.title)

        # Add date if available
        date_str = ""
        if conversation.created_at:
            try:
                # Try to parse various date formats
                if isinstance(conversation.created_at, (int, float)):
                    dt = datetime.fromtimestamp(conversation.created_at)
                else:
                    # Simple parse attempt
                    dt = datetime.fromisoformat(str(conversation.created_at).replace('Z', '+00:00'))
                date_str = f" - {dt.strftime('%Y%m%d_%H%M%S')}"
            except:
                pass  # Skip date if can't parse

        # Build filename
        subset_suffix = {'both': 'full', 'user': 'user', 'assistant': 'assistant'}[subset]
        base_name = f"{safe_title}{date_str} - {subset_suffix}"

        # Determine extension
        ext = {'markdown': 'md', 'md': 'md', 'text': 'txt', 'txt': 'txt', 'json': 'json'}[format]

        # Choose target directory
        if target_dir is None:
            target_dir = self.default_dir

        # Generate unique filename
        filepath = self._generate_unique_filename(base_name, target_dir, ext)

        # Generate content
        content = self._format_content(conversation, subset, format)

        # Write file
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)

        self.exported_files.append(filepath)

        if self.verbose:
            print(f"  Exported: {filepath.name}")

        return filepath

    def _format_content(self, conversation: Conversation, subset: str, format: str) -> str:
        """Format conversation content"""

        if format == 'json':
            messages = []
            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    messages.append({
                        'role': msg.role,
                        'content': msg.content,
                        'timestamp': msg.timestamp
                    })

            export_data = {
                'title': conversation.title,
                'platform': conversation.platform,
                'created_at': conversation.created_at,
                'messages': messages
            }
            return json.dumps(export_data, indent=2, ensure_ascii=False)

        elif format in ['markdown', 'md']:
            lines = [f"# {conversation.title}\n"]
            lines.append(f"**Platform:** {conversation.platform}\n")
            if conversation.created_at:
                lines.append(f"**Date:** {conversation.created_at}\n")
            lines.append("\n---\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "👤 User" if msg.role == 'user' else "🤖 Assistant"
                    lines.append(f"### {role_label}\n\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("---\n\n")

            return ''.join(lines)

        else:  # text format
            lines = [f"{conversation.title}\n"]
            lines.append(f"Platform: {conversation.platform}\n")
            lines.append("=" * 50 + "\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "USER" if msg.role == 'user' else "ASSISTANT"
                    lines.append(f"[{role_label}]:\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("-" * 30 + "\n\n")

            return ''.join(lines)

    def export_code_blocks(self, conversation: Conversation) -> List[Path]:
        """Export code blocks from conversation"""

        exported_code_files = []
        safe_title = self._sanitize_filename(conversation.title)

        code_num = 1
        for msg in conversation.messages:
            if msg.role == 'assistant':
                code_blocks = CodeExtractor.extract_code_blocks(msg.content)

                for code, language in code_blocks:
                    ext = CodeExtractor.LANGUAGE_EXTENSIONS.get(language, 'txt')
                    base_name = f"{safe_title} - code{code_num:02d}"

                    filepath = self._generate_unique_filename(base_name[:-4], self.code_dir, ext)

                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(code)

                    exported_code_files.append(filepath)
                    code_num += 1

                    if self.verbose:
                        print(f"    Code: {filepath.name}")

        return exported_code_files

# %% Keyword grouping
class KeywordGrouper:
    """Group conversations by keywords"""

    def __init__(self, conversations: List[Conversation]):
        self.conversations = conversations

    def group_by_keywords(self, keywords: List[str], unique: bool = False) -> Dict[str, List[Conversation]]:
        """Group conversations by specified keywords"""

        groups = defaultdict(list)

        for conv in self.conversations:
            matched_keywords = []

            # Check in title and messages
            full_text = conv.title.lower()
            for msg in conv.messages:
                full_text += " " + msg.content.lower()

            for keyword in keywords:
                if keyword.lower() in full_text:
                    matched_keywords.append(keyword)

            # Add to groups
            if matched_keywords:
                if unique:
                    # Add to first matched keyword only
                    groups[matched_keywords[0]].append(conv)
                else:
                    # Add to all matched keywords
                    for kw in matched_keywords:
                        groups[kw].append(conv)

        return dict(groups)

# %% File loading utilities
def detect_encoding(file_path: Path) -> str:
    """Detect file encoding with BOM sniffing"""

    with open(file_path, 'rb') as f:
        raw = f.read(4)

    # Check for BOM
    if raw.startswith(b'\xff\xfe\x00\x00'):
        return 'utf-32-le'
    elif raw.startswith(b'\x00\x00\xfe\xff'):
        return 'utf-32-be'
    elif raw.startswith(b'\xff\xfe'):
        return 'utf-16-le'
    elif raw.startswith(b'\xfe\xff'):
        return 'utf-16-be'
    elif raw.startswith(b'\xef\xbb\xbf'):
        return 'utf-8-sig'

    # Check for compressed files
    if raw.startswith(b'\x1f\x8b'):  # GZIP
        raise ValueError(f"File {file_path} appears to be gzipped. Please decompress it first.")
    elif raw.startswith(b'PK'):  # ZIP
        raise ValueError(f"File {file_path} appears to be zipped. Please extract it first.")

    # Try UTF-8
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            f.read(1024)
        return 'utf-8'
    except UnicodeDecodeError:
        pass

    # Fallback to Latin-1
    return 'latin-1'

def load_json_file(filepath: Path, verbose: bool = False) -> Optional[List[Conversation]]:
    """Load and parse a JSON file with encoding detection"""

    file_size_mb = filepath.stat().st_size / (1024**2)

    if verbose:
        print(f"Processing: {filepath.name} ({file_size_mb:.1f} MB)")

    # Detect encoding
    try:
        encoding = detect_encoding(filepath)
        if verbose and encoding != 'utf-8':
            print(f"  Detected encoding: {encoding}")
    except ValueError as e:
        print(f"  Error: {e}")
        return None

    # Check for large files
    if file_size_mb >= 1000:  # 1GB+
        try:
            import ijson
            if verbose:
                print(f"  Using streaming parser for large file")
            # Implement streaming parse (simplified for brevity)
            with open(filepath, 'rb') as f:
                data = json.load(f)
        except ImportError:
            print(f"  Warning: File ≥1GB and ijson not installed. Skipping to avoid memory issues.")
            print(f"  Install with: pip install ijson")
            return None
        except MemoryError:
            print(f"  Error: Out of memory processing {filepath.name}")
            return None
    else:
        # Regular loading
        with open(filepath, 'r', encoding=encoding, errors='replace') as f:
            data = json.load(f)

    # Parse conversations
    try:
        conversations = UniversalChatParser.parse(data)
        platform = conversations[0].platform if conversations else 'unknown'
        if verbose:
            print(f"  Platform: {platform}, Conversations: {len(conversations)}")
        return conversations
    except Exception as e:
        print(f"  Error parsing {filepath.name}: {e}")
        return None

# %% [markdown]
# ## Main Processing
#
# Run this cell to process all JSON files in `/content/input_data/`

# %% Main processing function
def process_chats(config: Dict):
    """Main processing function"""

    # Find all JSON files
    input_path = Path(config['input_root'])
    if not input_path.exists():
        print(f"Error: Input directory {input_path} does not exist")
        return

    json_files = list(input_path.glob('**/*.json')) + list(input_path.glob('**/*.jsonl'))

    if not json_files:
        print(f"No JSON files found in {input_path}")
        print("Nothing to do.")
        return

    print(f"Found {len(json_files)} JSON file(s) in {input_path}")

    if config['dry_run']:
        print("DRY RUN MODE - No files will be written")

    # Initialize exporter
    exporter = ChatExporter(config['output_root'], config['max_filename_len'], config['verbose'])

    # Process all files
    all_conversations = []
    platform_counts = defaultdict(int)

    for json_file in json_files:
        conversations = load_json_file(json_file, config['verbose'])
        if conversations:
            all_conversations.extend(conversations)
            for conv in conversations:
                platform_counts[conv.platform] += 1

    if not all_conversations:
        print("No conversations found in any files")
        return

    print(f"\nTotal conversations loaded: {len(all_conversations)}")
    for platform, count in platform_counts.items():
        print(f"  {platform}: {count}")

    if config['dry_run']:
        print("\nDry run complete. No files written.")
        return

    # Export conversations
    print("\nExporting conversations...")
    transcript_count = 0
    code_count = 0

    for conv in all_conversations:
        # Export transcript
        filepath = exporter.export_conversation(conv, config['subset'], config['format'])
        if filepath:
            transcript_count += 1

        # Export code if requested
        if config['export_code']:
            code_files = exporter.export_code_blocks(conv)
            code_count += len(code_files)

    # Handle keyword grouping
    group_count = 0
    if config['keywords']:
        print(f"\nGrouping by keywords: {', '.join(config['keywords'])}")
        grouper = KeywordGrouper(all_conversations)
        groups = grouper.group_by_keywords(config['keywords'], unique=(config['groups'] == 'unique'))

        for keyword, conversations in groups.items():
            keyword_dir = Path(config['output_root']) / f"keyword_{keyword}"
            keyword_dir.mkdir(parents=True, exist_ok=True)

            for conv in conversations:
                filepath = exporter.export_conversation(conv, config['subset'], config['format'], keyword_dir)
                if filepath:
                    group_count += 1

            print(f"  {keyword}: {len(conversations)} conversations")

    # Copy to Drive if requested
    if config['drive_out']:
        drive_path = Path(config['drive_out'])
        if drive_path.exists():
            print(f"\nCopying to Drive: {drive_path}")
            shutil.copytree(config['output_root'], drive_path / "chat_exports", dirs_exist_ok=True)
            print("  Copy complete")
        else:
            print(f"\nWarning: Drive path {drive_path} does not exist. Skipping copy.")

    # Summary
    print("\n" + "=" * 60)
    print("EXPORT SUMMARY")
    print("=" * 60)
    print(f"Files scanned: {len(json_files)}")
    print(f"Conversations parsed: {len(all_conversations)}")
    print(f"  ChatGPT: {platform_counts.get('chatgpt', 0)}")
    print(f"  Claude: {platform_counts.get('claude', 0)}")
    print(f"  Gemini: {platform_counts.get('gemini', 0)}")
    print(f"  Unknown: {platform_counts.get('unknown', 0)}")
    print(f"Transcripts exported: {transcript_count}")
    print(f"Code files exported: {code_count}")
    print(f"Keyword group outputs: {group_count}")
    print(f"\nOutput directory: {config['output_root']}")

# Run the processing
process_chats(CONFIG)

# %% [markdown]
# ## Check Output Files
#
# Run this cell to see what was exported:

# %% Check output
output_root = Path(CONFIG['output_root'])

if output_root.exists():
    print("📁 Output Directory Structure:")
    print("=" * 60)

    # Count files in each directory
    default_files = list((output_root / "default").glob("*")) if (output_root / "default").exists() else []
    code_files = list((output_root / "code_exports").glob("*")) if (output_root / "code_exports").exists() else []

    print(f"\n📄 Transcripts in default/: {len(default_files)}")
    if default_files[:5]:  # Show first 5
        for f in default_files[:5]:
            print(f"  • {f.name}")
        if len(default_files) > 5:
            print(f"  ... and {len(default_files) - 5} more")

    print(f"\n💻 Code files in code_exports/: {len(code_files)}")
    if code_files[:5]:  # Show first 5
        for f in code_files[:5]:
            print(f"  • {f.name}")
        if len(code_files) > 5:
            print(f"  ... and {len(code_files) - 5} more")

    # Check keyword directories
    keyword_dirs = [d for d in output_root.iterdir() if d.is_dir() and d.name.startswith("keyword_")]
    if keyword_dirs:
        print(f"\n🏷️ Keyword groups: {len(keyword_dirs)}")
        for d in keyword_dirs:
            files = list(d.glob("*"))
            print(f"  • {d.name}: {len(files)} files")
else:
    print("No output directory found. Run the processing cell first.")

# %% [markdown]
# ## Download Results
#
# Run this cell to create a ZIP file of all exports for download:

# %% Create downloadable ZIP
from google.colab import files
import zipfile

def create_download_zip():
    """Create a ZIP file of all exports"""

    output_root = Path(CONFIG['output_root'])
    if not output_root.exists():
        print("No exports found. Run the processing cell first.")
        return

    zip_path = "/content/chat_exports.zip"

    print("Creating ZIP file...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_root.rglob('*'):
            if file_path.is_file():
                arcname = file_path.relative_to(output_root.parent)
                zipf.write(file_path, arcname)

    print(f"✓ ZIP file created: {zip_path}")
    print(f"  Size: {Path(zip_path).stat().st_size / (1024**2):.1f} MB")

    # Download the file
    files.download(zip_path)
    print("📥 Download started...")

# Uncomment to download:
# create_download_zip()

# %% [markdown]
# ## Mount Google Drive (Optional)
#
# Run this cell if you want to save exports directly to Google Drive:

# %% Mount Google Drive (optional)
from google.colab import drive

# Uncomment to mount Drive:
# drive.mount('/content/drive')

# Then update CONFIG['drive_out'] and re-run processing:
# CONFIG['drive_out'] = '/content/drive/MyDrive/chat_exports'
# process_chats(CONFIG)

# %% [markdown]
# ## Test Suite
#
# Run these cells to verify the tool works correctly:

# %% Create test data
def create_test_data():
    """Create test files for verification"""

    input_dir = Path("/content/input_data")

    # Test 1: UTF-16 LE file with BOM
    print("Creating test files...")

    utf16_data = [{
        "title": "UTF16 Test Chat",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Test message"]},
                    "create_time": 1705159822
                }
            }
        }
    }]

    utf16_file = input_dir / "test_utf16.json"
    with open(utf16_file, 'w', encoding='utf-16-le') as f:
        f.write('\ufeff')  # BOM
        json.dump(utf16_data, f)

    # Test 2: Conversation with code blocks
    code_data = [{
        "title": "Code Examples",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Show me Python code"]}
                }
            },
            "2": {
                "message": {
                    "author": {"role": "assistant"},
                    "content": {"parts": [
                        "Here's Python:\n```python\ndef hello():\n    print('Hello')\n```"
                    ]}
                }
            }
        }
    }]

    with open(input_dir / "test_code.json", 'w') as f:
        json.dump(code_data, f)

    print("✓ Test files created")

# Run to create test data
create_test_data()

# Process test data
test_config = CONFIG.copy()
test_config['verbose'] = True
test_config['export_code'] = True
test_config['keywords'] = ['python', 'code']

print("\n" + "=" * 60)
print("RUNNING TESTS")
print("=" * 60)
process_chats(test_config)

# %% [markdown]
# ## Instructions
#
# ### Setup:
# 1. Run all cells in order
# 2. Place your JSON export files in `/content/input_data/`
#    - ChatGPT: `conversations.json`
#    - Claude: `conversations.json` (rename to avoid conflicts)
#    - Gemini: any JSON export file
#
# ### Configuration:
# - Modify the CONFIG dictionary in the Configuration cell
# - Key options:
#   - `subset`: "both", "user", or "assistant"
#   - `format`: "md", "txt", or "json"
#   - `export_code`: True/False
#   - `keywords`: List of keywords for grouping
#   - `drive_out`: Path to copy outputs to Drive
#
# ### Output Structure:
# ```
# /content/chat_exports/
# ├── default/                    # Main transcripts
# ├── code_exports/               # Extracted code blocks
# └── keyword_{keyword}/          # Grouped by keywords
# ```
#
# ### File Naming:
# - Transcripts: `<title> - <YYYYMMDD_HHMMSS> - <subset>.<ext>`
# - Code: `<title> - code<nn>.<ext>`
# - Collisions handled with `-<hash>` suffix
#
# ### Features:
# - ✅ Handles UTF-16/32 with BOM
# - ✅ Detects and warns about ZIP/GZIP files
# - ✅ Streams large files (1GB+) with ijson
# - ✅ Preserves all platform parsing logic
# - ✅ Exports to flat, sortable structure
# - ✅ Optional Drive backup

# Drive Export

In [ ]:
# Install pandoc (required by pypandoc) and pypandoc
!apt-get update && apt-get install -y pandoc 2>/dev/null && echo "✓ pandoc installed" || echo "⚠ pandoc not installed"
!pip install -q pypandoc 2>/dev/null && echo "✓ pypandoc installed" || echo "⚠ pypandoc not installed"

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,006 kB]
Get:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,797 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-back

In [ ]:
# Define the output directory and target Google Drive path
output_root = "/content/chat_exports"
drive_output_path = "/content/drive/MyDrive/chat_exports_download"

# Create the target directory in Google Drive if it doesn't exist
os.makedirs(drive_output_path, exist_ok=True)
print(f"Target Google Drive path created: {drive_output_path}")

# Walk through the output directory, convert and copy files
print("\nConverting and copying files to Google Drive...")
for subdir, _, files in os.walk(output_root):
    relative_subdir = os.path.relpath(subdir, output_root)
    target_subdir = os.path.join(drive_output_path, relative_subdir)
    os.makedirs(target_subdir, exist_ok=True)

    for file in files:
        file_path = os.path.join(subdir, file)
        file_name, file_ext = os.path.splitext(file)

        if file_ext.lower() == '.md':
            # Convert markdown to docx
            docx_file_name = file_name + '.docx'
            docx_file_path = os.path.join(target_subdir, docx_file_name)
            try:
                pypandoc.convert_file(file_path, 'docx', outputfile=docx_file_path)
                print(f"  Converted and copied: {file} -> {docx_file_name}")
            except Exception as e:
                print(f"  Error converting {file} to docx: {e}")
                # Copy the original markdown file if conversion fails
                shutil.copy2(file_path, target_subdir)
                print(f"  Copied original: {file}")
        else:
            # Copy other file types directly
            shutil.copy2(file_path, target_subdir)
            print(f"  Copied: {file}")

print("\nConversion and copying complete!")

Streaming output truncated to the last 5000 lines.
  Error converting Task List for UTO - 20240621_091612 - user.md to docx: name 'pypandoc' is not defined
  Copied original: Task List for UTO - 20240621_091612 - user.md
  Error converting Radius Equations in Desmos - 20250321_101653 - user.md to docx: name 'pypandoc' is not defined
  Copied original: Radius Equations in Desmos - 20250321_101653 - user.md
  Error converting Degrees of Freedom Explained - 20250119_083746 - user.md to docx: name 'pypandoc' is not defined
  Copied original: Degrees of Freedom Explained - 20250119_083746 - user.md
  Error converting Flutter Microservices Game Plan - 20230920_191239 - user.md to docx: name 'pypandoc' is not defined
  Copied original: Flutter Microservices Game Plan - 20230920_191239 - user.md
  Error converting Fire-Puppy Video Brief - 20250504_142644 - user.md to docx: name 'pypandoc' is not defined
  Copied original: Fire-Puppy Video Brief - 20250504_142644 - user.md
  Error converting Un

In [1]:
from google.colab import drive
import os
import shutil
# import pypandoc # Removed pypandoc import as direct conversion to Google Docs format is not supported

# Define directories
output_root = "/content/chat_exports"
merged_output_root = os.path.join(output_root, "merged")
drive_output_path = "/content/drive/MyDrive/chat_exports_download"

# Create merged output directory
os.makedirs(merged_output_root, exist_ok=True)
print(f"Merged output directory created: {merged_output_root}")

# Merge markdown files within each subdirectory
print("\nMerging markdown files...")
for subdir in os.listdir(output_root):
    subdir_path = os.path.join(output_root, subdir)
    # Exclude 'merged' and 'code_exports' directories
    if os.path.isdir(subdir_path) and subdir not in ["merged", "code_exports"]:
        merged_file_name = f"{subdir}_merged.md"
        merged_file_path = os.path.join(merged_output_root, merged_file_name)

        with open(merged_file_path, 'w', encoding='utf-8') as outfile:
            for root, _, files in os.walk(subdir_path):
                for file in files:
                    if file.endswith(".md"):
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8') as infile:
                            outfile.write(f"# File: {file}\n\n") # Add filename header
                            outfile.write(infile.read())
                            outfile.write("\n\n---\n\n") # Separator

        print(f"  Merged files in '{subdir}' into '{merged_file_name}'")


# Create the target directory in Google Drive if it doesn't exist
os.makedirs(drive_output_path, exist_ok=True)
print(f"\nTarget Google Drive path created: {drive_output_path}")

# Copy merged markdown files to Google Drive
print("\nCopying merged files to Google Drive...")
for file in os.listdir(merged_output_root):
    file_path = os.path.join(merged_output_root, file)
    # Only copy markdown files from the merged directory
    if file.endswith(".md"):
        try:
            shutil.copy2(file_path, drive_output_path)
            print(f"  Copied: {file}")
        except Exception as e:
            print(f"  Error copying {file} to Google Drive: {e}")


print("\nCopying complete!")
print("\nTo convert the markdown files to Google Docs format:")
print("1. Go to the Google Drive folder:")
print(f"   {drive_output_path}")
print("2. Right-click on each .md file.")
print("3. Select 'Open with' > 'Google Docs'.")
print("4. A new Google Doc will be created with the content of the markdown file.")

Merged output directory created: /content/chat_exports/merged

Merging markdown files...
  Merged files in 'keyword_algebra' into 'keyword_algebra_merged.md'
  Merged files in 'keyword_dimensionless' into 'keyword_dimensionless_merged.md'
  Merged files in 'keyword_tau' into 'keyword_tau_merged.md'
  Merged files in 'keyword_distinguish' into 'keyword_distinguish_merged.md'
  Merged files in '.ipynb_checkpoints' into '.ipynb_checkpoints_merged.md'
  Merged files in 'keyword_universe' into 'keyword_universe_merged.md'
  Merged files in 'keyword_3:4:5' into 'keyword_3:4:5_merged.md'
  Merged files in 'keyword_functional role' into 'keyword_functional role_merged.md'
  Merged files in 'keyword_temperature' into 'keyword_temperature_merged.md'
  Merged files in 'keyword_incompleteness' into 'keyword_incompleteness_merged.md'
  Merged files in 'default' into 'default_merged.md'
  Merged files in 'keyword_choice' into 'keyword_choice_merged.md'
  Merged files in 'keyword_geometry' into 'keyw

## New version

In [2]:
# %% [markdown]
# # Chat Export Tool for Google Colab
#
# Processes chat exports from ChatGPT, Claude, and Gemini platforms with CLI-like configuration.
#
# **Quick Start:**
# 1. Run all cells in order
# 2. Place your JSON files in `/content/input_data/`
# 3. Configure settings in the Config cell
# 4. Run the Process cell

# %% Install dependencies
!pip install -q ijson 2>/dev/null && echo "✓ ijson installed (for large file support)" || echo "⚠ ijson not installed"

# %% Import libraries and setup
import json
import os
import re
import sys
import hashlib
import gzip
import zipfile
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Union, Tuple, Any
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime
from glob import glob

# Create directories
Path("/content/input_data").mkdir(exist_ok=True)
Path("/content/chat_exports").mkdir(exist_ok=True)

print("✓ Directories created")
print("  Input:  /content/input_data/")
print("  Output: /content/chat_exports/")

# %% [markdown]
# ## Configuration
#
# Modify these settings as needed:

# %% Configuration settings
CONFIG = {
    "input_root": "/content/input_data",
    "output_root": "/content/chat_exports",
    "subset": "both",  # Options: "both", "user", "assistant"
    "format": "md",     # Options: "md", "txt", "json"
    "export_code": True,
    "keywords": [],     # e.g., ["python", "javascript", "api"]
    "groups": "non-unique",  # Options: "non-unique", "unique"
    "drive_out": None,  # e.g., "/content/drive/MyDrive/chat_exports"
    "max_filename_len": 160,
    "dry_run": False,
    "verbose": True
}

# Print current configuration
print("Current Configuration:")
print("-" * 40)
for key, value in CONFIG.items():
    print(f"{key:20s}: {value}")

# %% Data structures
@dataclass
class Message:
    """Represents a single message in a conversation"""
    role: str  # 'user' or 'assistant'
    content: str
    timestamp: Optional[str] = None
    metadata: Dict = field(default_factory=dict)

@dataclass
class Conversation:
    """Represents a complete conversation"""
    id: str
    title: str
    messages: List[Message]
    created_at: Optional[str] = None
    updated_at: Optional[str] = None
    platform: str = "unknown"
    metadata: Dict = field(default_factory=dict)

# %% Platform parsers
class ChatGPTParser:
    """Parser for ChatGPT conversation exports"""

    @staticmethod
    def parse(data: Dict) -> List[Conversation]:
        """Parse ChatGPT conversations.json format"""
        conversations = []

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')
            created_at = conv_data.get('create_time')
            updated_at = conv_data.get('update_time')

            messages = []
            mapping = conv_data.get('mapping', {})

            # Build message tree from mapping
            for node_id, node_data in mapping.items():
                message = node_data.get('message')
                if message and message.get('content'):
                    content_parts = message['content'].get('parts', [])
                    if content_parts:
                        content = '\n'.join(str(part) for part in content_parts)
                        role = message['author']['role']
                        # Map ChatGPT roles to our standard roles
                        if role == 'system':
                            continue  # Skip system messages
                        role = 'user' if role == 'user' else 'assistant'

                        msg = Message(
                            role=role,
                            content=content,
                            timestamp=message.get('create_time'),
                            metadata={'id': message.get('id')}
                        )
                        messages.append(msg)

            if messages:  # Only add if conversation has messages
                conversations.append(Conversation(
                    id=conv_id,
                    title=title,
                    messages=messages,
                    created_at=created_at,
                    updated_at=updated_at,
                    platform='chatgpt'
                ))

        return conversations

class ClaudeParser:
    """Parser for Claude conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Claude export format"""
        conversations = []

        # Handle both single conversation and multiple conversations
        if isinstance(data, dict):
            data = [data]

        for conv_data in data:
            # Try multiple possible formats
            conv_id = conv_data.get('id', conv_data.get('uuid', ''))
            title = conv_data.get('title', conv_data.get('name', 'Untitled'))

            messages = []
            # Check different possible message locations
            message_list = conv_data.get('messages', conv_data.get('chat_messages',
                                         conv_data.get('conversation', [])))

            for msg_data in message_list:
                # Handle different role naming conventions
                role = msg_data.get('role', msg_data.get('sender', ''))
                if role in ['human', 'user']:
                    role = 'user'
                elif role in ['assistant', 'claude', 'ai']:
                    role = 'assistant'
                else:
                    continue  # Skip unknown roles

                # Get content from various possible fields
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle content that might be in nested structure
                if isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)
                elif isinstance(content, dict):
                    content = content.get('text', str(content))

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp', msg_data.get('created_at')),
                        metadata=msg_data.get('metadata', {})
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='claude'
                ))

        return conversations

class GeminiParser:
    """Parser for Gemini conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Gemini export format"""
        conversations = []

        # Handle both single and multiple conversations
        if isinstance(data, dict):
            if 'conversations' in data:
                data = data['conversations']
            else:
                data = [data]

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')

            messages = []
            # Check for messages in different possible locations
            message_list = conv_data.get('messages', conv_data.get('entries',
                                         conv_data.get('chats', [])))

            for msg_data in message_list:
                # Determine role
                role = msg_data.get('role', msg_data.get('type', ''))
                if role in ['user', 'prompt']:
                    role = 'user'
                elif role in ['model', 'gemini', 'response', 'assistant']:
                    role = 'assistant'
                else:
                    continue

                # Extract content
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle nested content structures
                if isinstance(content, dict):
                    content = content.get('text', str(content))
                elif isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp'),
                        metadata=msg_data.get('metadata', {})
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='gemini'
                ))

        return conversations

class UniversalChatParser:
    """Universal parser that auto-detects platform format"""

    @staticmethod
    def detect_platform(data: Union[Dict, List]) -> str:
        """Detect which platform the data is from"""

        # Convert to dict if needed for checking
        sample = data[0] if isinstance(data, list) and data else data

        # ChatGPT indicators
        if isinstance(sample, dict):
            if 'mapping' in sample:
                return 'chatgpt'
            # Claude indicators
            if 'chat_messages' in sample or 'uuid' in sample:
                return 'claude'
            # Gemini indicators
            if 'entries' in sample or ('messages' in sample and
                any('model' in msg for msg in sample.get('messages', []))):
                return 'gemini'

        # Try to detect from message structure
        if 'messages' in sample:
            msgs = sample['messages']
            if msgs and isinstance(msgs[0], dict):
                if 'author' in msgs[0]:
                    return 'chatgpt'
                if 'sender' in msgs[0]:
                    return 'claude'
                if 'model' in msgs[0] or 'type' in msgs[0]:
                    return 'gemini'

        return 'unknown'

    @staticmethod
    def parse(data: Union[Dict, List], platform: Optional[str] = None) -> List[Conversation]:
        """Parse chat data from any supported platform"""

        if platform is None:
            platform = UniversalChatParser.detect_platform(data)

        parsers = {
            'chatgpt': ChatGPTParser(),
            'claude': ClaudeParser(),
            'gemini': GeminiParser()
        }

        parser = parsers.get(platform, ClaudeParser())  # Default to Claude parser
        return parser.parse(data)

# %% Code extraction
class CodeExtractor:
    """Extract code blocks from messages"""

    LANGUAGE_EXTENSIONS = {
        'python': '.py', 'javascript': '.js', 'typescript': '.ts',
        'java': '.java', 'cpp': '.cpp', 'c': '.c', 'csharp': '.cs',
        'html': '.html', 'css': '.css', 'sql': '.sql', 'bash': '.sh',
        'shell': '.sh', 'yaml': '.yml', 'json': '.json', 'xml': '.xml',
        'markdown': '.md', 'rust': '.rs', 'go': '.go', 'ruby': '.rb',
        'php': '.php', 'swift': '.swift', 'kotlin': '.kt', 'r': '.r'
    }

    @staticmethod
    def extract_code_blocks(content: str) -> List[Tuple[str, str]]:
        """Extract code blocks from content
        Returns: List of (code, language) tuples"""

        code_blocks = []

        # Pattern for ```language\ncode\n```
        pattern = r'```(\w+)?\n?(.*?)```'
        matches = re.findall(pattern, content, re.DOTALL)

        for language, code in matches:
            language = language.lower() if language else 'text'
            code_blocks.append((code.strip(), language))

        # Also check for inline code if no blocks found
        if not code_blocks:
            # Look for common code patterns
            inline_pattern = r'`([^`]+)`'
            inline_matches = re.findall(inline_pattern, content)
            for code in inline_matches:
                if len(code) > 50:  # Only consider substantial inline code
                    code_blocks.append((code, 'text'))

        return code_blocks

# %% Export functionality
class ChatExporter:
    """Export conversations in various formats"""

    def __init__(self, output_root: str, max_filename_len: int = 160, verbose: bool = False):
        self.output_root = Path(output_root)
        self.max_filename_len = max_filename_len
        self.verbose = verbose
        self.exported_files = []
        self.filename_cache = {}  # Track used filenames for collision detection

        # Create output directories
        self.default_dir = self.output_root / "default"
        self.code_dir = self.output_root / "code_exports"

        self.default_dir.mkdir(parents=True, exist_ok=True)
        self.code_dir.mkdir(parents=True, exist_ok=True)

    def _sanitize_filename(self, filename: str) -> str:
        """Sanitize filename for filesystem"""
        # Remove/replace invalid characters
        invalid_chars = '<>:"/\\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')
        # Limit length
        return filename[:self.max_filename_len]

    def _generate_unique_filename(self, base_name: str, directory: Path, ext: str) -> Path:
        """Generate unique filename with collision handling"""
        # Check if filename already used
        filepath = directory / f"{base_name}.{ext}"

        if filepath not in self.filename_cache:
            self.filename_cache[filepath] = True
            return filepath

        # Add hash for uniqueness
        for i in range(1, 100):
            hash_suffix = hashlib.md5(f"{base_name}{i}".encode()).hexdigest()[:6]
            unique_name = f"{base_name}-{hash_suffix}"
            filepath = directory / f"{unique_name}.{ext}"

            if filepath not in self.filename_cache:
                self.filename_cache[filepath] = True
                return filepath

        # Fallback with timestamp
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        filepath = directory / f"{base_name}-{timestamp}.{ext}"
        self.filename_cache[filepath] = True
        return filepath

    def export_conversation(self, conversation: Conversation, subset: str, format: str,
                          target_dir: Optional[Path] = None) -> Optional[Path]:
        """Export a single conversation"""

        # Generate filename components
        safe_title = self._sanitize_filename(conversation.title)

        # Add date if available
        date_str = ""
        if conversation.created_at:
            try:
                # Try to parse various date formats
                if isinstance(conversation.created_at, (int, float)):
                    dt = datetime.fromtimestamp(conversation.created_at)
                else:
                    # Simple parse attempt
                    dt = datetime.fromisoformat(str(conversation.created_at).replace('Z', '+00:00'))
                date_str = f" - {dt.strftime('%Y%m%d_%H%M%S')}"
            except:
                pass  # Skip date if can't parse

        # Build filename
        subset_suffix = {'both': 'full', 'user': 'user', 'assistant': 'assistant'}[subset]
        base_name = f"{safe_title}{date_str} - {subset_suffix}"

        # Determine extension
        ext = {'markdown': 'md', 'md': 'md', 'text': 'txt', 'txt': 'txt', 'json': 'json'}[format]

        # Choose target directory
        if target_dir is None:
            target_dir = self.default_dir

        # Generate unique filename
        filepath = self._generate_unique_filename(base_name, target_dir, ext)

        # Generate content
        content = self._format_content(conversation, subset, format)

        # Write file
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)

        self.exported_files.append(filepath)

        if self.verbose:
            print(f"  Exported: {filepath.name}")

        return filepath

    def _format_content(self, conversation: Conversation, subset: str, format: str) -> str:
        """Format conversation content"""

        if format == 'json':
            messages = []
            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    messages.append({
                        'role': msg.role,
                        'content': msg.content,
                        'timestamp': msg.timestamp
                    })

            export_data = {
                'title': conversation.title,
                'platform': conversation.platform,
                'created_at': conversation.created_at,
                'messages': messages
            }
            return json.dumps(export_data, indent=2, ensure_ascii=False)

        elif format in ['markdown', 'md']:
            lines = [f"# {conversation.title}\n"]
            lines.append(f"**Platform:** {conversation.platform}\n")
            if conversation.created_at:
                lines.append(f"**Date:** {conversation.created_at}\n")
            lines.append("\n---\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "👤 User" if msg.role == 'user' else "🤖 Assistant"
                    lines.append(f"### {role_label}\n\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("---\n\n")

            return ''.join(lines)

        else:  # text format
            lines = [f"{conversation.title}\n"]
            lines.append(f"Platform: {conversation.platform}\n")
            lines.append("=" * 50 + "\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "USER" if msg.role == 'user' else "ASSISTANT"
                    lines.append(f"[{role_label}]:\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("-" * 30 + "\n\n")

            return ''.join(lines)

    def export_code_blocks(self, conversation: Conversation) -> List[Path]:
        """Export code blocks from conversation"""

        exported_code_files = []
        safe_title = self._sanitize_filename(conversation.title)

        code_num = 1
        for msg in conversation.messages:
            if msg.role == 'assistant':
                code_blocks = CodeExtractor.extract_code_blocks(msg.content)

                for code, language in code_blocks:
                    ext = CodeExtractor.LANGUAGE_EXTENSIONS.get(language, 'txt')
                    base_name = f"{safe_title} - code{code_num:02d}"

                    filepath = self._generate_unique_filename(base_name[:-4], self.code_dir, ext)

                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(code)

                    exported_code_files.append(filepath)
                    code_num += 1

                    if self.verbose:
                        print(f"    Code: {filepath.name}")

        return exported_code_files

# %% Keyword grouping
class KeywordGrouper:
    """Group conversations by keywords"""

    def __init__(self, conversations: List[Conversation]):
        self.conversations = conversations

    def group_by_keywords(self, keywords: List[str], unique: bool = False) -> Dict[str, List[Conversation]]:
        """Group conversations by specified keywords"""

        groups = defaultdict(list)

        for conv in self.conversations:
            matched_keywords = []

            # Check in title and messages
            full_text = conv.title.lower()
            for msg in conv.messages:
                full_text += " " + msg.content.lower()

            for keyword in keywords:
                if keyword.lower() in full_text:
                    matched_keywords.append(keyword)

            # Add to groups
            if matched_keywords:
                if unique:
                    # Add to first matched keyword only
                    groups[matched_keywords[0]].append(conv)
                else:
                    # Add to all matched keywords
                    for kw in matched_keywords:
                        groups[kw].append(conv)

        return dict(groups)

# %% File loading utilities
def detect_encoding(file_path: Path) -> str:
    """Detect file encoding with BOM sniffing"""

    with open(file_path, 'rb') as f:
        raw = f.read(4)

    # Check for BOM
    if raw.startswith(b'\xff\xfe\x00\x00'):
        return 'utf-32-le'
    elif raw.startswith(b'\x00\x00\xfe\xff'):
        return 'utf-32-be'
    elif raw.startswith(b'\xff\xfe'):
        return 'utf-16-le'
    elif raw.startswith(b'\xfe\xff'):
        return 'utf-16-be'
    elif raw.startswith(b'\xef\xbb\xbf'):
        return 'utf-8-sig'

    # Check for compressed files
    if raw.startswith(b'\x1f\x8b'):  # GZIP
        raise ValueError(f"File {file_path} appears to be gzipped. Please decompress it first.")
    elif raw.startswith(b'PK'):  # ZIP
        raise ValueError(f"File {file_path} appears to be zipped. Please extract it first.")

    # Try UTF-8
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            f.read(1024)
        return 'utf-8'
    except UnicodeDecodeError:
        pass

    # Fallback to Latin-1
    return 'latin-1'

def load_json_file(filepath: Path, verbose: bool = False) -> Optional[List[Conversation]]:
    """Load and parse a JSON file with encoding detection"""

    file_size_mb = filepath.stat().st_size / (1024**2)

    if verbose:
        print(f"Processing: {filepath.name} ({file_size_mb:.1f} MB)")

    # Detect encoding
    try:
        encoding = detect_encoding(filepath)
        if verbose and encoding != 'utf-8':
            print(f"  Detected encoding: {encoding}")
    except ValueError as e:
        print(f"  Error: {e}")
        return None

    # Check for large files
    if file_size_mb >= 1000:  # 1GB+
        try:
            import ijson
            if verbose:
                print(f"  Using streaming parser for large file")
            # Implement streaming parse (simplified for brevity)
            with open(filepath, 'rb') as f:
                data = json.load(f)
        except ImportError:
            print(f"  Warning: File ≥1GB and ijson not installed. Skipping to avoid memory issues.")
            print(f"  Install with: pip install ijson")
            return None
        except MemoryError:
            print(f"  Error: Out of memory processing {filepath.name}")
            return None
    else:
        # Regular loading
        with open(filepath, 'r', encoding=encoding, errors='replace') as f:
            data = json.load(f)

    # Parse conversations
    try:
        conversations = UniversalChatParser.parse(data)
        platform = conversations[0].platform if conversations else 'unknown'
        if verbose:
            print(f"  Platform: {platform}, Conversations: {len(conversations)}")
        return conversations
    except Exception as e:
        print(f"  Error parsing {filepath.name}: {e}")
        return None

# %% [markdown]
# ## Main Processing
#
# Run this cell to process all JSON files in `/content/input_data/`

# %% Main processing function
def process_chats(config: Dict):
    """Main processing function"""

    # Find all JSON files
    input_path = Path(config['input_root'])
    if not input_path.exists():
        print(f"Error: Input directory {input_path} does not exist")
        return

    json_files = list(input_path.glob('**/*.json')) + list(input_path.glob('**/*.jsonl'))

    if not json_files:
        print(f"No JSON files found in {input_path}")
        print("Nothing to do.")
        return

    print(f"Found {len(json_files)} JSON file(s) in {input_path}")

    if config['dry_run']:
        print("DRY RUN MODE - No files will be written")

    # Initialize exporter
    exporter = ChatExporter(config['output_root'], config['max_filename_len'], config['verbose'])

    # Process all files
    all_conversations = []
    platform_counts = defaultdict(int)

    for json_file in json_files:
        conversations = load_json_file(json_file, config['verbose'])
        if conversations:
            all_conversations.extend(conversations)
            for conv in conversations:
                platform_counts[conv.platform] += 1

    if not all_conversations:
        print("No conversations found in any files")
        return

    print(f"\nTotal conversations loaded: {len(all_conversations)}")
    for platform, count in platform_counts.items():
        print(f"  {platform}: {count}")

    if config['dry_run']:
        print("\nDry run complete. No files written.")
        return

    # Export conversations
    print("\nExporting conversations...")
    transcript_count = 0
    code_count = 0

    for conv in all_conversations:
        # Export transcript
        filepath = exporter.export_conversation(conv, config['subset'], config['format'])
        if filepath:
            transcript_count += 1

        # Export code if requested
        if config['export_code']:
            code_files = exporter.export_code_blocks(conv)
            code_count += len(code_files)

    # Handle keyword grouping
    group_count = 0
    if config['keywords']:
        print(f"\nGrouping by keywords: {', '.join(config['keywords'])}")
        grouper = KeywordGrouper(all_conversations)
        groups = grouper.group_by_keywords(config['keywords'], unique=(config['groups'] == 'unique'))

        for keyword, conversations in groups.items():
            keyword_dir = Path(config['output_root']) / f"keyword_{keyword}"
            keyword_dir.mkdir(parents=True, exist_ok=True)

            for conv in conversations:
                filepath = exporter.export_conversation(conv, config['subset'], config['format'], keyword_dir)
                if filepath:
                    group_count += 1

            print(f"  {keyword}: {len(conversations)} conversations")

    # Copy to Drive if requested
    if config['drive_out']:
        drive_path = Path(config['drive_out'])
        if drive_path.exists():
            print(f"\nCopying to Drive: {drive_path}")
            shutil.copytree(config['output_root'], drive_path / "chat_exports", dirs_exist_ok=True)
            print("  Copy complete")
        else:
            print(f"\nWarning: Drive path {drive_path} does not exist. Skipping copy.")

    # Summary
    print("\n" + "=" * 60)
    print("EXPORT SUMMARY")
    print("=" * 60)
    print(f"Files scanned: {len(json_files)}")
    print(f"Conversations parsed: {len(all_conversations)}")
    print(f"  ChatGPT: {platform_counts.get('chatgpt', 0)}")
    print(f"  Claude: {platform_counts.get('claude', 0)}")
    print(f"  Gemini: {platform_counts.get('gemini', 0)}")
    print(f"  Unknown: {platform_counts.get('unknown', 0)}")
    print(f"Transcripts exported: {transcript_count}")
    print(f"Code files exported: {code_count}")
    print(f"Keyword group outputs: {group_count}")
    print(f"\nOutput directory: {config['output_root']}")

# Run the processing
process_chats(CONFIG)

# %% [markdown]
# ## Check Output Files
#
# Run this cell to see what was exported:

# %% Check output
output_root = Path(CONFIG['output_root'])

if output_root.exists():
    print("📁 Output Directory Structure:")
    print("=" * 60)

    # Count files in each directory
    default_files = list((output_root / "default").glob("*")) if (output_root / "default").exists() else []
    code_files = list((output_root / "code_exports").glob("*")) if (output_root / "code_exports").exists() else []

    print(f"\n📄 Transcripts in default/: {len(default_files)}")
    if default_files[:5]:  # Show first 5
        for f in default_files[:5]:
            print(f"  • {f.name}")
        if len(default_files) > 5:
            print(f"  ... and {len(default_files) - 5} more")

    print(f"\n💻 Code files in code_exports/: {len(code_files)}")
    if code_files[:5]:  # Show first 5
        for f in code_files[:5]:
            print(f"  • {f.name}")
        if len(code_files) > 5:
            print(f"  ... and {len(code_files) - 5} more")

    # Check keyword directories
    keyword_dirs = [d for d in output_root.iterdir() if d.is_dir() and d.name.startswith("keyword_")]
    if keyword_dirs:
        print(f"\n🏷️ Keyword groups: {len(keyword_dirs)}")
        for d in keyword_dirs:
            files = list(d.glob("*"))
            print(f"  • {d.name}: {len(files)} files")
else:
    print("No output directory found. Run the processing cell first.")

# %% [markdown]
# ## Download Results
#
# Run this cell to create a ZIP file of all exports for download:

# %% Create downloadable ZIP
from google.colab import files
import zipfile

def create_download_zip():
    """Create a ZIP file of all exports"""

    output_root = Path(CONFIG['output_root'])
    if not output_root.exists():
        print("No exports found. Run the processing cell first.")
        return

    zip_path = "/content/chat_exports.zip"

    print("Creating ZIP file...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_root.rglob('*'):
            if file_path.is_file():
                arcname = file_path.relative_to(output_root.parent)
                zipf.write(file_path, arcname)

    print(f"✓ ZIP file created: {zip_path}")
    print(f"  Size: {Path(zip_path).stat().st_size / (1024**2):.1f} MB")

    # Download the file
    files.download(zip_path)
    print("📥 Download started...")

# Uncomment to download:
# create_download_zip()

# %% [markdown]
# ## Mount Google Drive (Optional)
#
# Run this cell if you want to save exports directly to Google Drive:

# %% Mount Google Drive (optional)
from google.colab import drive

# Uncomment to mount Drive:
# drive.mount('/content/drive')

# Then update CONFIG['drive_out'] and re-run processing:
# CONFIG['drive_out'] = '/content/drive/MyDrive/chat_exports'
# process_chats(CONFIG)

# %% [markdown]
# ## Test Suite
#
# Run these cells to verify the tool works correctly:

# %% Create test data
def create_test_data():
    """Create test files for verification"""

    input_dir = Path("/content/input_data")

    # Test 1: UTF-16 LE file with BOM
    print("Creating test files...")

    utf16_data = [{
        "title": "UTF16 Test Chat",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Test message"]},
                    "create_time": 1705159822
                }
            }
        }
    }]

    utf16_file = input_dir / "test_utf16.json"
    with open(utf16_file, 'w', encoding='utf-16-le') as f:
        f.write('\ufeff')  # BOM
        json.dump(utf16_data, f)

    # Test 2: Conversation with code blocks
    code_data = [{
        "title": "Code Examples",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Show me Python code"]}
                }
            },
            "2": {
                "message": {
                    "author": {"role": "assistant"},
                    "content": {"parts": [
                        "Here's Python:\n```python\ndef hello():\n    print('Hello')\n```"
                    ]}
                }
            }
        }
    }]

    with open(input_dir / "test_code.json", 'w') as f:
        json.dump(code_data, f)

    print("✓ Test files created")

# Run to create test data
create_test_data()

# Process test data
test_config = CONFIG.copy()
test_config['verbose'] = True
test_config['export_code'] = True
test_config['keywords'] = ['python', 'code']

print("\n" + "=" * 60)
print("RUNNING TESTS")
print("=" * 60)
process_chats(test_config)

# %% [markdown]
# ## Instructions
#
# ### Setup:
# 1. Run all cells in order
# 2. Place your JSON export files in `/content/input_data/`
#    - ChatGPT: `conversations.json`
#    - Claude: `conversations.json` (rename to avoid conflicts)
#    - Gemini: any JSON export file
#
# ### Configuration:
# - Modify the CONFIG dictionary in the Configuration cell
# - Key options:
#   - `subset`: "both", "user", or "assistant"
#   - `format`: "md", "txt", or "json"
#   - `export_code`: True/False
#   - `keywords`: List of keywords for grouping
#   - `drive_out`: Path to copy outputs to Drive
#
# ### Output Structure:
# ```
# /content/chat_exports/
# ├── default/                    # Main transcripts
# ├── code_exports/               # Extracted code blocks
# └── keyword_{keyword}/          # Grouped by keywords
# ```
#
# ### File Naming:
# - Transcripts: `<title> - <YYYYMMDD_HHMMSS> - <subset>.<ext>`
# - Code: `<title> - code<nn>.<ext>`
# - Collisions handled with `-<hash>` suffix
#
# ### Features:
# - ✅ Handles UTF-16/32 with BOM
# - ✅ Detects and warns about ZIP/GZIP files
# - ✅ Streams large files (1GB+) with ijson
# - ✅ Preserves all platform parsing logic
# - ✅ Exports to flat, sortable structure
# - ✅ Optional Drive backup

✓ ijson installed (for large file support)
✓ Directories created
  Input:  /content/input_data/
  Output: /content/chat_exports/
Current Configuration:
----------------------------------------
input_root          : /content/input_data
output_root         : /content/chat_exports
subset              : both
format              : md
export_code         : True
keywords            : []
groups              : non-unique
drive_out           : None
max_filename_len    : 160
dry_run             : False
verbose             : True
Found 4 JSON file(s) in /content/input_data
Processing: claude_conversations.json (1114.2 MB)
  Detected encoding: utf-16-le
  Using streaming parser for large file


KeyboardInterrupt: 

In [3]:
# %% [markdown]
# # Chat Export Tool for Google Colab v8
#
# Processes chat exports from ChatGPT, Claude, and Gemini platforms with CLI-like configuration.
#
# **v8 Features:**
# - Sources Mode: Creates ≤150 stitched documents of user text with provenance
# - Keyword aggregation into single files
# - Code-only export mode
# - Improved robustness for large files and formats
#
# **Quick Start:**
# 1. Run all cells in order
# 2. Place your JSON files in `/content/input_data/`
# 3. Configure settings in the Config cell
# 4. Run the Process cell

# %% Install dependencies
!pip install -q ijson 2>/dev/null && echo "✓ ijson installed (for large file support)" || echo "⚠ ijson not installed"

# %% Import libraries and setup
import json
import os
import re
import sys
import hashlib
import gzip
import zipfile
import shutil
import csv
from pathlib import Path
from typing import Dict, List, Optional, Union, Tuple, Any, Set
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from datetime import datetime
from glob import glob

# Create directories
Path("/content/input_data").mkdir(exist_ok=True)
Path("/content/chat_exports").mkdir(exist_ok=True)

print("✓ Directories created")
print("  Input:  /content/input_data/")
print("  Output: /content/chat_exports/")

# %% [markdown]
# ## Configuration
#
# Modify these settings as needed:

# %% Configuration settings
CONFIG = {
    "input_root": "/content/input_data",
    "output_root": "/content/chat_exports",
    "subset": "both",  # Options: "both", "user", "assistant"
    "format": "md",     # Options: "md", "txt", "json"
    "export_code": True,
    "keywords": [],     # e.g., ["python", "javascript", "api"]
    "groups": "non-unique",  # Options: "non-unique", "unique"
    "drive_out": None,  # e.g., "/content/drive/MyDrive/chat_exports"
    "max_filename_len": 160,
    "dry_run": False,
    "verbose": True,
    # v8 additions
    "build_sources": True,            # toggle Sources Mode
    "sources_cap": 150,               # hard cap on source files
    "include_assistant_context": False,  # add brief assistant quotes as blockquotes
    "min_user_chars": 400,            # threshold for "extended" user segments
    "similarity_threshold": 0.35,     # Jaccard overlap for attach/near-dup
    "code_only": False,               # if True, export only code blocks
    "aggregate_keyword_files": False  # write keyword_{kw}.md aggregates
}

# Print current configuration
print("Current Configuration:")
print("-" * 40)
for key, value in CONFIG.items():
    print(f"{key:28s}: {value}")

# %% Data structures
@dataclass
class Message:
    """Represents a single message in a conversation"""
    role: str  # 'user' or 'assistant'
    content: str
    timestamp: Optional[str] = None
    metadata: Dict = field(default_factory=dict)
    index: Optional[int] = None  # v8: track message index

@dataclass
class Conversation:
    """Represents a complete conversation"""
    id: str
    title: str
    messages: List[Message]
    created_at: Optional[str] = None
    updated_at: Optional[str] = None
    platform: str = "unknown"
    metadata: Dict = field(default_factory=dict)

# v8: Sources Mode data structures
@dataclass
class Segment:
    """Represents a user text segment for sources mode"""
    conversation_id: str
    conversation_title: str
    message_index: int
    content: str
    timestamp: Optional[str] = None
    assistant_context: Optional[str] = None
    content_hash: str = ""

    def __post_init__(self):
        if not self.content_hash:
            self.content_hash = hashlib.md5(self.content.encode()).hexdigest()

# %% Platform parsers
class ChatGPTParser:
    """Parser for ChatGPT conversation exports"""

    @staticmethod
    def parse(data: Dict) -> List[Conversation]:
        """Parse ChatGPT conversations.json format"""
        conversations = []

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')
            created_at = conv_data.get('create_time')
            updated_at = conv_data.get('update_time')

            messages = []

            # v8: Try new format first (messages array)
            if 'messages' in conv_data and conv_data['messages']:
                message_list = conv_data['messages']
                for idx, msg in enumerate(message_list):
                    if msg.get('role') == 'system':
                        continue  # Skip system messages

                    content = msg.get('content', '')
                    if isinstance(content, dict):
                        content = content.get('text', '')

                    if content:
                        role = 'user' if msg.get('role') == 'user' else 'assistant'
                        messages.append(Message(
                            role=role,
                            content=str(content),
                            timestamp=msg.get('timestamp'),
                            metadata={'id': msg.get('id')},
                            index=idx
                        ))

            # Fallback to mapping format
            elif 'mapping' in conv_data:
                mapping = conv_data.get('mapping', {})

                # Build message tree from mapping
                for node_id, node_data in mapping.items():
                    message = node_data.get('message')
                    if message and message.get('content'):
                        content_parts = message['content'].get('parts', [])
                        if content_parts:
                            content = '\n'.join(str(part) for part in content_parts)
                            role = message['author']['role']
                            # Map ChatGPT roles to our standard roles
                            if role == 'system':
                                continue  # Skip system messages
                            role = 'user' if role == 'user' else 'assistant'

                            msg = Message(
                                role=role,
                                content=content,
                                timestamp=message.get('create_time'),
                                metadata={'id': message.get('id')}
                            )
                            messages.append(msg)

            if messages:  # Only add if conversation has messages
                # Add indices if not present
                for idx, msg in enumerate(messages):
                    if msg.index is None:
                        msg.index = idx

                conversations.append(Conversation(
                    id=conv_id,
                    title=title,
                    messages=messages,
                    created_at=created_at,
                    updated_at=updated_at,
                    platform='chatgpt'
                ))

        return conversations

class ClaudeParser:
    """Parser for Claude conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Claude export format"""
        conversations = []

        # Handle both single conversation and multiple conversations
        if isinstance(data, dict):
            data = [data]

        for conv_data in data:
            # Try multiple possible formats
            conv_id = conv_data.get('id', conv_data.get('uuid', ''))
            title = conv_data.get('title', conv_data.get('name', 'Untitled'))

            messages = []
            # Check different possible message locations
            message_list = conv_data.get('messages', conv_data.get('chat_messages',
                                         conv_data.get('conversation', [])))

            for idx, msg_data in enumerate(message_list):
                # Handle different role naming conventions
                role = msg_data.get('role', msg_data.get('sender', ''))
                if role in ['human', 'user']:
                    role = 'user'
                elif role in ['assistant', 'claude', 'ai']:
                    role = 'assistant'
                else:
                    continue  # Skip unknown roles

                # Get content from various possible fields
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle content that might be in nested structure
                if isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)
                elif isinstance(content, dict):
                    content = content.get('text', str(content))

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp', msg_data.get('created_at')),
                        metadata=msg_data.get('metadata', {}),
                        index=idx
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='claude'
                ))

        return conversations

class GeminiParser:
    """Parser for Gemini conversation exports"""

    @staticmethod
    def parse(data: Union[Dict, List]) -> List[Conversation]:
        """Parse Gemini export format"""
        conversations = []

        # Handle both single and multiple conversations
        if isinstance(data, dict):
            if 'conversations' in data:
                data = data['conversations']
            else:
                data = [data]

        for conv_data in data:
            conv_id = conv_data.get('id', '')
            title = conv_data.get('title', 'Untitled')

            messages = []
            # Check for messages in different possible locations
            message_list = conv_data.get('messages', conv_data.get('entries',
                                         conv_data.get('chats', [])))

            for idx, msg_data in enumerate(message_list):
                # Determine role
                role = msg_data.get('role', msg_data.get('type', ''))
                if role in ['user', 'prompt']:
                    role = 'user'
                elif role in ['model', 'gemini', 'response', 'assistant']:
                    role = 'assistant'
                else:
                    continue

                # Extract content
                content = msg_data.get('content', msg_data.get('text',
                                       msg_data.get('message', '')))

                # Handle nested content structures
                if isinstance(content, dict):
                    content = content.get('text', str(content))
                elif isinstance(content, list):
                    content = '\n'.join(str(item) for item in content)

                if content:
                    messages.append(Message(
                        role=role,
                        content=str(content),
                        timestamp=msg_data.get('timestamp'),
                        metadata=msg_data.get('metadata', {}),
                        index=idx
                    ))

            if messages:
                conversations.append(Conversation(
                    id=conv_id or hashlib.md5(title.encode()).hexdigest()[:8],
                    title=title,
                    messages=messages,
                    created_at=conv_data.get('created_at'),
                    updated_at=conv_data.get('updated_at'),
                    platform='gemini'
                ))

        return conversations

class UniversalChatParser:
    """Universal parser that auto-detects platform format"""

    @staticmethod
    def detect_platform(data: Union[Dict, List]) -> str:
        """Detect which platform the data is from"""

        # Convert to dict if needed for checking
        sample = data[0] if isinstance(data, list) and data else data

        # ChatGPT indicators
        if isinstance(sample, dict):
            if 'mapping' in sample or ('messages' in sample and
                                       any('author' in msg or 'role' in msg
                                           for msg in sample.get('messages', [])
                                           if isinstance(msg, dict))):
                return 'chatgpt'
            # Claude indicators
            if 'chat_messages' in sample or 'uuid' in sample:
                return 'claude'
            # Gemini indicators
            if 'entries' in sample or ('messages' in sample and
                any('model' in msg for msg in sample.get('messages', []))):
                return 'gemini'

        # Try to detect from message structure
        if 'messages' in sample:
            msgs = sample['messages']
            if msgs and isinstance(msgs[0], dict):
                if 'author' in msgs[0]:
                    return 'chatgpt'
                if 'sender' in msgs[0]:
                    return 'claude'
                if 'model' in msgs[0] or 'type' in msgs[0]:
                    return 'gemini'

        # v8: Return unknown cleanly instead of defaulting
        return 'unknown'

    @staticmethod
    def parse(data: Union[Dict, List], platform: Optional[str] = None) -> List[Conversation]:
        """Parse chat data from any supported platform"""

        if platform is None:
            platform = UniversalChatParser.detect_platform(data)

        parsers = {
            'chatgpt': ChatGPTParser(),
            'claude': ClaudeParser(),
            'gemini': GeminiParser()
        }

        # v8: Handle unknown platform more gracefully
        if platform not in parsers:
            print(f"  Warning: Unknown platform detected. Attempting generic parse.")
            # Try Claude parser as most flexible fallback
            return ClaudeParser().parse(data)

        return parsers[platform].parse(data)

# %% Code extraction
class CodeExtractor:
    """Extract code blocks from messages"""

    # v8: Fixed - no leading dots in extensions
    LANGUAGE_EXTENSIONS = {
        'python': 'py', 'javascript': 'js', 'typescript': 'ts',
        'java': 'java', 'cpp': 'cpp', 'c': 'c', 'csharp': 'cs',
        'html': 'html', 'css': 'css', 'sql': 'sql', 'bash': 'sh',
        'shell': 'sh', 'yaml': 'yml', 'json': 'json', 'xml': 'xml',
        'markdown': 'md', 'rust': 'rs', 'go': 'go', 'ruby': 'rb',
        'php': 'php', 'swift': 'swift', 'kotlin': 'kt', 'r': 'r'
    }

    @staticmethod
    def extract_code_blocks(content: str) -> List[Tuple[str, str]]:
        """Extract code blocks from content
        Returns: List of (code, language) tuples"""

        code_blocks = []

        # v8: Improved pattern - non-greedy, handles more language tags
        pattern = r'```([a-zA-Z0-9_+\-\.]*)\n(.*?)```'
        matches = re.findall(pattern, content, re.DOTALL)

        for language, code in matches:
            language = language.lower() if language else 'text'
            code_blocks.append((code.strip(), language))

        # Also check for inline code if no blocks found
        if not code_blocks:
            # Look for common code patterns
            inline_pattern = r'`([^`]+)`'
            inline_matches = re.findall(inline_pattern, content)
            for code in inline_matches:
                if len(code) > 50:  # Only consider substantial inline code
                    code_blocks.append((code, 'text'))

        return code_blocks

# %% Export functionality
class ChatExporter:
    """Export conversations in various formats"""

    def __init__(self, output_root: str, max_filename_len: int = 160, verbose: bool = False):
        self.output_root = Path(output_root)
        self.max_filename_len = max_filename_len
        self.verbose = verbose
        self.exported_files = []
        self.filename_cache = {}  # Track used filenames for collision detection

        # Create output directories
        self.default_dir = self.output_root / "default"
        self.code_dir = self.output_root / "code_exports"
        self.sources_dir = self.output_root / "sources"  # v8
        self.meta_dir = self.output_root / "meta"  # v8

        self.default_dir.mkdir(parents=True, exist_ok=True)
        self.code_dir.mkdir(parents=True, exist_ok=True)

    def _sanitize_filename(self, filename: str) -> str:
        """Sanitize filename for filesystem"""
        # Remove/replace invalid characters
        invalid_chars = '<>:"/\\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')
        # Limit length
        return filename[:self.max_filename_len]

    def _generate_unique_filename(self, base_name: str, directory: Path, ext: str) -> Path:
        """Generate unique filename with collision handling"""
        # Check if filename already used
        filepath = directory / f"{base_name}.{ext}"

        if filepath not in self.filename_cache:
            self.filename_cache[filepath] = True
            return filepath

        # Add hash for uniqueness
        for i in range(1, 100):
            hash_suffix = hashlib.md5(f"{base_name}{i}".encode()).hexdigest()[:6]
            unique_name = f"{base_name}-{hash_suffix}"
            filepath = directory / f"{unique_name}.{ext}"

            if filepath not in self.filename_cache:
                self.filename_cache[filepath] = True
                return filepath

        # Fallback with timestamp
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        filepath = directory / f"{base_name}-{timestamp}.{ext}"
        self.filename_cache[filepath] = True
        return filepath

    def export_conversation(self, conversation: Conversation, subset: str, format: str,
                          target_dir: Optional[Path] = None) -> Optional[Path]:
        """Export a single conversation"""

        # Generate filename components
        safe_title = self._sanitize_filename(conversation.title)

        # Add date if available
        date_str = ""
        if conversation.created_at:
            try:
                # Try to parse various date formats
                if isinstance(conversation.created_at, (int, float)):
                    dt = datetime.fromtimestamp(conversation.created_at)
                else:
                    # Simple parse attempt
                    dt = datetime.fromisoformat(str(conversation.created_at).replace('Z', '+00:00'))
                date_str = f" - {dt.strftime('%Y%m%d_%H%M%S')}"
            except:
                pass  # Skip date if can't parse

        # Build filename
        subset_suffix = {'both': 'full', 'user': 'user', 'assistant': 'assistant'}[subset]
        base_name = f"{safe_title}{date_str} - {subset_suffix}"

        # Determine extension
        ext = {'markdown': 'md', 'md': 'md', 'text': 'txt', 'txt': 'txt', 'json': 'json'}[format]

        # Choose target directory
        if target_dir is None:
            target_dir = self.default_dir

        # Generate unique filename
        filepath = self._generate_unique_filename(base_name, target_dir, ext)

        # Generate content
        content = self._format_content(conversation, subset, format)

        # Write file
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)

        self.exported_files.append(filepath)

        if self.verbose:
            print(f"  Exported: {filepath.name}")

        return filepath

    def _format_content(self, conversation: Conversation, subset: str, format: str) -> str:
        """Format conversation content"""

        if format == 'json':
            messages = []
            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    messages.append({
                        'role': msg.role,
                        'content': msg.content,
                        'timestamp': msg.timestamp
                    })

            export_data = {
                'title': conversation.title,
                'platform': conversation.platform,
                'created_at': conversation.created_at,
                'messages': messages
            }
            return json.dumps(export_data, indent=2, ensure_ascii=False)

        elif format in ['markdown', 'md']:
            lines = [f"# {conversation.title}\n"]
            lines.append(f"**Platform:** {conversation.platform}\n")
            if conversation.created_at:
                lines.append(f"**Date:** {conversation.created_at}\n")
            lines.append("\n---\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "👤 User" if msg.role == 'user' else "🤖 Assistant"
                    lines.append(f"### {role_label}\n\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("---\n\n")

            return ''.join(lines)

        else:  # text format
            lines = [f"{conversation.title}\n"]
            lines.append(f"Platform: {conversation.platform}\n")
            lines.append("=" * 50 + "\n\n")

            for msg in conversation.messages:
                if subset == 'both' or subset == msg.role:
                    role_label = "USER" if msg.role == 'user' else "ASSISTANT"
                    lines.append(f"[{role_label}]:\n")
                    lines.append(f"{msg.content}\n\n")
                    lines.append("-" * 30 + "\n\n")

            return ''.join(lines)

    def export_code_blocks(self, conversation: Conversation) -> List[Path]:
        """Export code blocks from conversation"""

        exported_code_files = []
        safe_title = self._sanitize_filename(conversation.title)

        code_num = 1
        for msg in conversation.messages:
            if msg.role == 'assistant':
                code_blocks = CodeExtractor.extract_code_blocks(msg.content)

                for code, language in code_blocks:
                    # v8: Fixed - use extension without dot, remove basename slicing
                    ext = CodeExtractor.LANGUAGE_EXTENSIONS.get(language, 'txt')
                    base_name = f"{safe_title} - code{code_num:02d}"

                    filepath = self._generate_unique_filename(base_name, self.code_dir, ext)

                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(code)

                    exported_code_files.append(filepath)
                    code_num += 1

                    if self.verbose:
                        print(f"    Code: {filepath.name}")

        return exported_code_files

# %% Keyword grouping
class KeywordGrouper:
    """Group conversations by keywords"""

    def __init__(self, conversations: List[Conversation]):
        self.conversations = conversations

    def group_by_keywords(self, keywords: List[str], unique: bool = False) -> Dict[str, List[Conversation]]:
        """Group conversations by specified keywords"""

        groups = defaultdict(list)

        for conv in self.conversations:
            matched_keywords = []

            # Check in title and messages
            full_text = conv.title.lower()
            for msg in conv.messages:
                full_text += " " + msg.content.lower()

            for keyword in keywords:
                if keyword.lower() in full_text:
                    matched_keywords.append(keyword)

            # Add to groups
            if matched_keywords:
                if unique:
                    # Add to first matched keyword only
                    groups[matched_keywords[0]].append(conv)
                else:
                    # Add to all matched keywords
                    for kw in matched_keywords:
                        groups[kw].append(conv)

        return dict(groups)

# %% v8: Sources Mode
class SourcesBuilder:
    """Build source documents from user text segments"""

    def __init__(self, conversations: List[Conversation], config: Dict):
        self.conversations = conversations
        self.config = config
        self.sources: Dict[str, List[Segment]] = {}
        self.source_metadata: Dict[str, Dict] = {}

    def _normalize_title(self, title: str) -> str:
        """Normalize title for grouping similar conversations"""
        # Remove dates, numbers, special chars
        normalized = re.sub(r'\d{4}-\d{2}-\d{2}', '', title)
        normalized = re.sub(r'\d+', '', normalized)
        normalized = re.sub(r'[^\w\s]', ' ', normalized)
        normalized = ' '.join(normalized.split()).lower().strip()
        return normalized or 'untitled'

    def _tokenize(self, text: str) -> Set[str]:
        """Simple word tokenization for Jaccard similarity"""
        words = re.findall(r'\b\w+\b', text.lower())
        return set(words)

    def _jaccard_similarity(self, text1: str, text2: str) -> float:
        """Calculate Jaccard similarity between two texts"""
        tokens1 = self._tokenize(text1)
        tokens2 = self._tokenize(text2)
        if not tokens1 or not tokens2:
            return 0.0
        intersection = tokens1 & tokens2
        union = tokens1 | tokens2
        return len(intersection) / len(union) if union else 0.0

    def extract_segments(self) -> List[Segment]:
        """Extract extended user segments from all conversations"""
        segments = []

        for conv in self.conversations:
            for msg in conv.messages:
                if msg.role == 'user' and len(msg.content) >= self.config['min_user_chars']:
                    # Get assistant context if enabled
                    assistant_context = None
                    if self.config['include_assistant_context']:
                        # Find next assistant message
                        idx = msg.index or 0
                        if idx + 1 < len(conv.messages):
                            next_msg = conv.messages[idx + 1]
                            if next_msg.role == 'assistant':
                                # Take first 200 chars as context
                                assistant_context = next_msg.content[:200]
                                if len(next_msg.content) > 200:
                                    assistant_context += "..."

                    segment = Segment(
                        conversation_id=conv.id,
                        conversation_title=conv.title,
                        message_index=msg.index or 0,
                        content=msg.content,
                        timestamp=msg.timestamp or conv.created_at,
                        assistant_context=assistant_context
                    )
                    segments.append(segment)

        return segments

    def build_sources(self) -> Tuple[Dict[str, List[Segment]], List[str]]:
        """Build source documents with cap enforcement"""
        segments = self.extract_segments()
        if not segments:
            return {}, []

        # Group by normalized title and calculate sizes
        title_groups = defaultdict(list)
        for seg in segments:
            norm_title = self._normalize_title(seg.conversation_title)
            title_groups[norm_title].append(seg)

        # Calculate aggregate sizes and select seeds
        title_sizes = {}
        for norm_title, segs in title_groups.items():
            total_chars = sum(len(s.content) for s in segs)
            title_sizes[norm_title] = total_chars

        # Select top sources by size
        sorted_titles = sorted(title_sizes.keys(), key=lambda t: title_sizes[t], reverse=True)
        seed_titles = sorted_titles[:self.config['sources_cap']]
        candidate_titles = sorted_titles[self.config['sources_cap']:]

        # Initialize sources with seeds
        sources = {title: title_groups[title] for title in seed_titles}

        # Greedy attach remaining segments
        threshold = self.config['similarity_threshold']
        for norm_title in candidate_titles:
            for seg in title_groups[norm_title]:
                best_match = None
                best_score = 0

                # Find best matching source
                for source_title, source_segs in sources.items():
                    # Calculate similarity to source segments
                    for source_seg in source_segs[:5]:  # Check first 5 for efficiency
                        score = self._jaccard_similarity(seg.content, source_seg.content)
                        if score > best_score:
                            best_score = score
                            best_match = source_title

                # Attach if similar enough
                if best_score >= threshold and best_match:
                    sources[best_match].append(seg)

        # Deduplicate within each source
        for title in sources:
            sources[title] = self._deduplicate_segments(sources[title])

        return sources, candidate_titles

    def _deduplicate_segments(self, segments: List[Segment]) -> List[Segment]:
        """Remove duplicate and near-duplicate segments"""
        if not segments:
            return []

        unique_segments = []
        seen_hashes = set()

        # Sort by length (prefer longer versions)
        segments_sorted = sorted(segments, key=lambda s: len(s.content), reverse=True)

        for seg in segments_sorted:
            # Check exact duplicate
            if seg.content_hash in seen_hashes:
                continue

            # Check near-duplicate
            is_duplicate = False
            for unique_seg in unique_segments:
                similarity = self._jaccard_similarity(seg.content, unique_seg.content)
                if similarity > 0.9:  # Very high similarity threshold for dedup
                    is_duplicate = True
                    break

            if not is_duplicate:
                unique_segments.append(seg)
                seen_hashes.add(seg.content_hash)

        # Sort chronologically
        unique_segments.sort(key=lambda s: (s.timestamp or '', s.message_index))

        return unique_segments

    def export_sources(self, exporter: ChatExporter) -> Tuple[int, List[Dict]]:
        """Export source documents and metadata"""
        sources, candidates = self.build_sources()

        if not sources:
            return 0, []

        # Create directories
        exporter.sources_dir.mkdir(exist_ok=True)
        exporter.meta_dir.mkdir(exist_ok=True)

        sources_metadata = []
        exported_count = 0

        for idx, (norm_title, segments) in enumerate(sources.items()):
            if not segments:
                continue

            # Generate source filename
            canonical_title = segments[0].conversation_title  # Use first segment's title
            safe_title = exporter._sanitize_filename(canonical_title)

            # Try to get earliest timestamp
            timestamps = [s.timestamp for s in segments if s.timestamp]
            if timestamps:
                try:
                    min_ts = min(timestamps)
                    if isinstance(min_ts, (int, float)):
                        dt = datetime.fromtimestamp(min_ts)
                    else:
                        dt = datetime.fromisoformat(str(min_ts).replace('Z', '+00:00'))
                    date_suffix = f" - {dt.strftime('%Y%m%d_%H%M%S')}"
                except:
                    date_suffix = ""
            else:
                date_suffix = ""

            base_name = f"{safe_title}{date_suffix}"
            filepath = exporter._generate_unique_filename(base_name, exporter.sources_dir, 'md')

            # Build content
            content = self._format_source_document(canonical_title, segments)

            # Write file
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)

            exported_count += 1

            # Collect metadata
            sources_metadata.append({
                'source_id': f"src_{idx:03d}",
                'canonical_title': canonical_title,
                'n_segments': len(segments),
                'n_chars': sum(len(s.content) for s in segments),
                'created_ts_min': min(timestamps) if timestamps else None,
                'created_ts_max': max(timestamps) if timestamps else None,
                'provenance_count': len(set(s.conversation_id for s in segments)),
                'is_candidate': False,
                'filepath': str(filepath.name)
            })

            if exporter.verbose:
                print(f"  Source: {filepath.name} ({len(segments)} segments)")

        # Mark candidates
        for candidate_title in candidates:
            sources_metadata.append({
                'source_id': f"cand_{len(sources_metadata):03d}",
                'canonical_title': candidate_title,
                'n_segments': 0,
                'n_chars': 0,
                'created_ts_min': None,
                'created_ts_max': None,
                'provenance_count': 0,
                'is_candidate': True,
                'filepath': None
            })

        # Write metadata CSVs
        self._write_metadata(sources, sources_metadata, exporter.meta_dir)

        return exported_count, sources_metadata

    def _format_source_document(self, title: str, segments: List[Segment]) -> str:
        """Format a source document with segments and provenance"""
        lines = [f"# {title}\n\n"]
        lines.append(f"*Source document compiled from {len(segments)} user text segments*\n\n")
        lines.append("---\n\n")

        # Add segments
        for seg in segments:
            lines.append(f"{seg.content}\n\n")

            if seg.assistant_context:
                lines.append(f"> **Assistant context:** {seg.assistant_context}\n\n")

            lines.append("---\n\n")

        # Add provenance section
        lines.append("## Provenance\n\n")

        # Group by conversation
        conv_groups = defaultdict(list)
        for seg in segments:
            conv_groups[seg.conversation_id].append(seg)

        for conv_id, conv_segs in conv_groups.items():
            conv_title = conv_segs[0].conversation_title
            indices = [s.message_index for s in conv_segs]

            lines.append(f"- **{conv_title}** (ID: {conv_id})\n")
            lines.append(f"  - Messages: {', '.join(map(str, indices))}\n")

            timestamps = [s.timestamp for s in conv_segs if s.timestamp]
            if timestamps:
                lines.append(f"  - Timestamps: {', '.join(timestamps[:3])}")
                if len(timestamps) > 3:
                    lines.append(f" ... ({len(timestamps)} total)")
                lines.append("\n")

        return ''.join(lines)

    def _write_metadata(self, sources: Dict, sources_metadata: List[Dict], meta_dir: Path):
        """Write metadata CSV files"""

        # Write sources index
        sources_csv = meta_dir / 'sources_index.csv'
        with open(sources_csv, 'w', newline='', encoding='utf-8') as f:
            if sources_metadata:
                writer = csv.DictWriter(f, fieldnames=sources_metadata[0].keys())
                writer.writeheader()
                writer.writerows(sources_metadata)

        # Write segments index
        segments_data = []
        for source_title, segments in sources.items():
            for seg in segments:
                segments_data.append({
                    'source_title': source_title,
                    'conversation_id': seg.conversation_id,
                    'conversation_title': seg.conversation_title,
                    'message_index': seg.message_index,
                    'content_preview': seg.content[:100] + '...' if len(seg.content) > 100 else seg.content,
                    'timestamp': seg.timestamp,
                    'content_hash': seg.content_hash
                })

        segments_csv = meta_dir / 'segments_index.csv'
        if segments_data:
            with open(segments_csv, 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=segments_data[0].keys())
                writer.writeheader()
                writer.writerows(segments_data)

# %% File loading utilities
def detect_encoding(file_path: Path) -> str:
    """Detect file encoding with BOM sniffing"""

    with open(file_path, 'rb') as f:
        raw = f.read(4)

    # Check for BOM
    if raw.startswith(b'\xff\xfe\x00\x00'):
        return 'utf-32-le'
    elif raw.startswith(b'\x00\x00\xfe\xff'):
        return 'utf-32-be'
    elif raw.startswith(b'\xff\xfe'):
        return 'utf-16-le'
    elif raw.startswith(b'\xfe\xff'):
        return 'utf-16-be'
    elif raw.startswith(b'\xef\xbb\xbf'):
        return 'utf-8-sig'

    # Check for compressed files
    if raw.startswith(b'\x1f\x8b'):  # GZIP
        raise ValueError(f"File {file_path} appears to be gzipped. Please decompress it first.")
    elif raw.startswith(b'PK'):  # ZIP
        raise ValueError(f"File {file_path} appears to be zipped. Please extract it first.")

    # Try UTF-8
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            f.read(1024)
        return 'utf-8'
    except UnicodeDecodeError:
        pass

    # Fallback to Latin-1
    return 'latin-1'

def load_json_file(filepath: Path, verbose: bool = False) -> Optional[List[Conversation]]:
    """Load and parse a JSON file with encoding detection"""

    file_size_mb = filepath.stat().st_size / (1024**2)

    if verbose:
        print(f"Processing: {filepath.name} ({file_size_mb:.1f} MB)")

    # v8: Check for JSONL
    if filepath.suffix == '.jsonl':
        if verbose:
            print(f"  Detected JSONL format")
        conversations = []

        encoding = detect_encoding(filepath)
        with open(filepath, 'r', encoding=encoding, errors='replace') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                    convs = UniversalChatParser.parse(data)
                    conversations.extend(convs)
                except Exception as e:
                    print(f"  Error parsing line {line_num}: {e}")

        if verbose and conversations:
            platform = conversations[0].platform if conversations else 'unknown'
            print(f"  Platform: {platform}, Conversations: {len(conversations)}")
        return conversations if conversations else None

    # Detect encoding
    try:
        encoding = detect_encoding(filepath)
        if verbose and encoding != 'utf-8':
            print(f"  Detected encoding: {encoding}")
    except ValueError as e:
        print(f"  Error: {e}")
        return None

    # Check for large files
    if file_size_mb >= 1000:  # 1GB+
        try:
            import ijson
            if verbose:
                print(f"  Using streaming parser for large file")

            # v8: Real streaming implementation
            conversations = []
            with open(filepath, 'rb') as f:
                # Try to parse as array of items
                try:
                    parser = ijson.items(f, 'item')
                    for item in parser:
                        convs = UniversalChatParser.parse(item)
                        conversations.extend(convs)
                except:
                    # Fallback if not array
                    f.seek(0)
                    data = json.load(f)
                    conversations = UniversalChatParser.parse(data)

            if verbose and conversations:
                platform = conversations[0].platform if conversations else 'unknown'
                print(f"  Platform: {platform}, Conversations: {len(conversations)}")
            return conversations if conversations else None

        except ImportError:
            print(f"  Warning: File ≥1GB and ijson not installed. Skipping to avoid memory issues.")
            print(f"  Install with: pip install ijson")
            return None
        except MemoryError:
            print(f"  Error: Out of memory processing {filepath.name}")
            return None
    else:
        # Regular loading
        with open(filepath, 'r', encoding=encoding, errors='replace') as f:
            data = json.load(f)

    # Parse conversations
    try:
        conversations = UniversalChatParser.parse(data)
        platform = conversations[0].platform if conversations else 'unknown'
        if verbose:
            print(f"  Platform: {platform}, Conversations: {len(conversations)}")
        return conversations
    except Exception as e:
        print(f"  Error parsing {filepath.name}: {e}")
        return None

# %% [markdown]
# ## Main Processing
#
# Run this cell to process all JSON files in `/content/input_data/`

# %% Main processing function
def process_chats(config: Dict) -> Tuple[List[Conversation], Dict[str, int]]:
    """Main processing function

    v8: Returns (all_conversations, platform_counts) for reuse
    """

    # Find all JSON files
    input_path = Path(config['input_root'])
    if not input_path.exists():
        print(f"Error: Input directory {input_path} does not exist")
        return [], {}

    json_files = list(input_path.glob('**/*.json')) + list(input_path.glob('**/*.jsonl'))

    if not json_files:
        print(f"No JSON files found in {input_path}")
        print("Nothing to do.")
        return [], {}

    print(f"Found {len(json_files)} JSON file(s) in {input_path}")

    if config['dry_run']:
        print("DRY RUN MODE - No files will be written")

    # Initialize exporter
    exporter = ChatExporter(config['output_root'], config['max_filename_len'], config['verbose'])

    # Process all files
    all_conversations = []
    platform_counts = defaultdict(int)

    for json_file in json_files:
        conversations = load_json_file(json_file, config['verbose'])
        if conversations:
            all_conversations.extend(conversations)
            for conv in conversations:
                platform_counts[conv.platform] += 1

    if not all_conversations:
        print("No conversations found in any files")
        return [], {}

    print(f"\nTotal conversations loaded: {len(all_conversations)}")
    for platform, count in platform_counts.items():
        print(f"  {platform}: {count}")

    if config['dry_run']:
        print("\nDry run complete. No files written.")
        return all_conversations, dict(platform_counts)

    # v8: Check for code_only mode
    if config['code_only']:
        print("\nCode-only mode - Exporting only code blocks...")
        code_count = 0
        for conv in all_conversations:
            code_files = exporter.export_code_blocks(conv)
            code_count += len(code_files)

        print(f"\nCode files exported: {code_count}")
        print(f"Output directory: {config['output_root']}/code_exports")
        return all_conversations, dict(platform_counts)

    # Export conversations
    print("\nExporting conversations...")
    transcript_count = 0
    code_count = 0

    for conv in all_conversations:
        # Export transcript
        filepath = exporter.export_conversation(conv, config['subset'], config['format'])
        if filepath:
            transcript_count += 1

        # Export code if requested
        if config['export_code']:
            code_files = exporter.export_code_blocks(conv)
            code_count += len(code_files)

    # Handle keyword grouping
    group_count = 0
    keyword_aggregate_count = 0
    if config['keywords']:
        print(f"\nGrouping by keywords: {', '.join(config['keywords'])}")
        grouper = KeywordGrouper(all_conversations)
        groups = grouper.group_by_keywords(config['keywords'], unique=(config['groups'] == 'unique'))

        for keyword, conversations in groups.items():
            keyword_dir = Path(config['output_root']) / f"keyword_{keyword}"
            keyword_dir.mkdir(parents=True, exist_ok=True)

            for conv in conversations:
                filepath = exporter.export_conversation(conv, config['subset'], config['format'], keyword_dir)
                if filepath:
                    group_count += 1

            # v8: Aggregate keyword files
            if config['aggregate_keyword_files']:
                aggregate_file = keyword_dir / f"keyword_{keyword}.md"
                aggregate_content = _create_keyword_aggregate(keyword, conversations, config['subset'])
                with open(aggregate_file, 'w', encoding='utf-8') as f:
                    f.write(aggregate_content)
                keyword_aggregate_count += 1
                if config['verbose']:
                    print(f"  Aggregate: {aggregate_file.name}")

            print(f"  {keyword}: {len(conversations)} conversations")

    # v8: Build sources if enabled
    sources_count = 0
    sources_metadata = []
    if config['build_sources']:
        print("\nBuilding source documents...")
        builder = SourcesBuilder(all_conversations, config)
        sources_count, sources_metadata = builder.export_sources(exporter)
        print(f"Sources exported: {sources_count} (cap={config['sources_cap']})")

    # Copy to Drive if requested
    if config['drive_out']:
        drive_path = Path(config['drive_out'])
        if drive_path.exists():
            print(f"\nCopying to Drive: {drive_path}")
            shutil.copytree(config['output_root'], drive_path / "chat_exports", dirs_exist_ok=True)
            print("  Copy complete")
        else:
            print(f"\nWarning: Drive path {drive_path} does not exist. Skipping copy.")

    # Summary
    print("\n" + "=" * 60)
    print("EXPORT SUMMARY")
    print("=" * 60)
    print(f"Files scanned: {len(json_files)}")
    print(f"Conversations parsed: {len(all_conversations)}")
    print(f"  ChatGPT: {platform_counts.get('chatgpt', 0)}")
    print(f"  Claude: {platform_counts.get('claude', 0)}")
    print(f"  Gemini: {platform_counts.get('gemini', 0)}")
    print(f"  Unknown: {platform_counts.get('unknown', 0)}")
    print(f"Transcripts exported: {transcript_count}")
    print(f"Code files exported: {code_count}")
    print(f"Keyword group outputs: {group_count}")
    if config['aggregate_keyword_files']:
        print(f"Keyword aggregates: {keyword_aggregate_count}")
    if config['build_sources']:
        print(f"Sources exported: {sources_count} (cap={config['sources_cap']})")
        if sources_metadata:
            candidates = sum(1 for s in sources_metadata if s.get('is_candidate'))
            if candidates:
                print(f"  Candidate sources (not exported): {candidates}")
        print(f"  Metadata: {config['output_root']}/meta/")
    print(f"\nOutput directory: {config['output_root']}")

    return all_conversations, dict(platform_counts)

def _create_keyword_aggregate(keyword: str, conversations: List[Conversation], subset: str) -> str:
    """Create aggregated markdown for keyword group"""
    lines = [f"# Keyword: {keyword}\n\n"]
    lines.append(f"*Aggregated from {len(conversations)} conversations*\n\n")
    lines.append("---\n\n")

    # Sort by date if available
    sorted_convs = sorted(conversations,
                         key=lambda c: c.created_at or '',
                         reverse=False)

    for conv in sorted_convs:
        lines.append(f"## {conv.title}\n\n")
        if conv.created_at:
            lines.append(f"*Date: {conv.created_at}*\n\n")

        for msg in conv.messages:
            if subset == 'both' or subset == msg.role:
                role_label = "👤 User" if msg.role == 'user' else "🤖 Assistant"
                lines.append(f"**{role_label}:**\n\n")
                lines.append(f"{msg.content}\n\n")

        lines.append("---\n\n")

    # Add provenance footer
    lines.append("## Provenance\n\n")
    for conv in sorted_convs:
        lines.append(f"- {conv.title} (ID: {conv.id})\n")

    return ''.join(lines)

# Run the processing
all_conversations, platform_counts = process_chats(CONFIG)

# %% [markdown]
# ## Check Output Files
#
# Run this cell to see what was exported:

# %% Check output
output_root = Path(CONFIG['output_root'])

if output_root.exists():
    print("📁 Output Directory Structure:")
    print("=" * 60)

    # Count files in each directory
    default_files = list((output_root / "default").glob("*")) if (output_root / "default").exists() else []
    code_files = list((output_root / "code_exports").glob("*")) if (output_root / "code_exports").exists() else []
    source_files = list((output_root / "sources").glob("*")) if (output_root / "sources").exists() else []
    meta_files = list((output_root / "meta").glob("*")) if (output_root / "meta").exists() else []

    print(f"\n📄 Transcripts in default/: {len(default_files)}")
    if default_files[:5]:  # Show first 5
        for f in default_files[:5]:
            print(f"  • {f.name}")
        if len(default_files) > 5:
            print(f"  ... and {len(default_files) - 5} more")

    print(f"\n💻 Code files in code_exports/: {len(code_files)}")
    if code_files[:5]:  # Show first 5
        for f in code_files[:5]:
            print(f"  • {f.name}")
        if len(code_files) > 5:
            print(f"  ... and {len(code_files) - 5} more")

    # v8: Check sources
    print(f"\n📚 Source documents in sources/: {len(source_files)}")
    if source_files[:5]:
        for f in source_files[:5]:
            print(f"  • {f.name}")
        if len(source_files) > 5:
            print(f"  ... and {len(source_files) - 5} more")

    # v8: Check metadata
    print(f"\n📊 Metadata files in meta/: {len(meta_files)}")
    for f in meta_files:
        print(f"  • {f.name}")

    # Check keyword directories
    keyword_dirs = [d for d in output_root.iterdir() if d.is_dir() and d.name.startswith("keyword_")]
    if keyword_dirs:
        print(f"\n🏷️ Keyword groups: {len(keyword_dirs)}")
        for d in keyword_dirs:
            files = list(d.glob("*"))
            print(f"  • {d.name}: {len(files)} files")
            # Check for aggregate file
            aggregate = d / f"{d.name}.md"
            if aggregate.exists():
                print(f"    - Has aggregate file")
else:
    print("No output directory found. Run the processing cell first.")

# %% [markdown]
# ## Download Results
#
# Run this cell to create a ZIP file of all exports for download:

# %% Create downloadable ZIP
from google.colab import files
import zipfile

def create_download_zip():
    """Create a ZIP file of all exports"""

    output_root = Path(CONFIG['output_root'])
    if not output_root.exists():
        print("No exports found. Run the processing cell first.")
        return

    zip_path = "/content/chat_exports.zip"

    print("Creating ZIP file...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_root.rglob('*'):
            if file_path.is_file():
                arcname = file_path.relative_to(output_root.parent)
                zipf.write(file_path, arcname)

    print(f"✓ ZIP file created: {zip_path}")
    print(f"  Size: {Path(zip_path).stat().st_size / (1024**2):.1f} MB")

    # Download the file
    files.download(zip_path)
    print("📥 Download started...")

# Uncomment to download:
# create_download_zip()

# %% [markdown]
# ## Mount Google Drive (Optional)
#
# Run this cell if you want to save exports directly to Google Drive:

# %% Mount Google Drive (optional)
from google.colab import drive

# Uncomment to mount Drive:
# drive.mount('/content/drive')

# Then update CONFIG['drive_out'] and re-run processing:
# CONFIG['drive_out'] = '/content/drive/MyDrive/chat_exports'
# process_chats(CONFIG)

# %% [markdown]
# ## Test Suite
#
# Run these cells to verify the tool works correctly:

# %% Create test data
def create_test_data():
    """Create test files for verification"""

    input_dir = Path("/content/input_data")

    # Test 1: UTF-16 LE file with BOM
    print("Creating test files...")

    utf16_data = [{
        "title": "UTF16 Test Chat",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Test message"]},
                    "create_time": 1705159822
                }
            }
        }
    }]

    utf16_file = input_dir / "test_utf16.json"
    with open(utf16_file, 'w', encoding='utf-16-le') as f:
        f.write('\ufeff')  # BOM
        json.dump(utf16_data, f)

    # Test 2: Conversation with code blocks
    code_data = [{
        "title": "Code Examples",
        "mapping": {
            "1": {
                "message": {
                    "author": {"role": "user"},
                    "content": {"parts": ["Show me Python code"]}
                }
            },
            "2": {
                "message": {
                    "author": {"role": "assistant"},
                    "content": {"parts": [
                        "Here's Python:\n```python\ndef hello():\n    print('Hello')\n```"
                    ]}
                }
            }
        }
    }]

    with open(input_dir / "test_code.json", 'w') as f:
        json.dump(code_data, f)

    # v8 Test 3: Extended user messages for sources
    sources_data = [{
        "title": "Research Discussion",
        "messages": [
            {
                "role": "user",
                "content": "This is a long user message about research topics. " * 50,  # 500+ chars
                "timestamp": "2024-01-15T10:00:00Z"
            },
            {
                "role": "assistant",
                "content": "Here's my response about the research."
            },
            {
                "role": "user",
                "content": "Another extended discussion about methodology. " * 40,
                "timestamp": "2024-01-15T10:05:00Z"
            }
        ]
    }]

    with open(input_dir / "test_sources.json", 'w') as f:
        json.dump(sources_data, f)

    print("✓ Test files created")

# Run to create test data
create_test_data()

# Process test data
test_config = CONFIG.copy()
test_config['verbose'] = True
test_config['export_code'] = True
test_config['keywords'] = ['python', 'code', 'research']
test_config['aggregate_keyword_files'] = True
test_config['build_sources'] = True

print("\n" + "=" * 60)
print("RUNNING TESTS")
print("=" * 60)
test_conversations, test_counts = process_chats(test_config)

# %% [markdown]
# ## Instructions
#
# ### v8.1 Features:
# - **Sources Mode**: Creates ≤150 stitched documents from extended user text
# - **Keyword Aggregation**: Single markdown file per keyword group
# - **Code-Only Mode**: Export only code blocks when `code_only: True`
# - **Improved Robustness**: Better handling of large files, JSONL, and various formats
# - **Fixed Issues**:
#   - Timestamp normalization prevents TypeError with mixed formats
#   - ChatGPT mapping reconstructs proper thread order
#   - Streaming handles object-wrapped arrays
#   - Similarity checks up to 20 segments (sampled if more)
#   - JSONL supports per-message format
#   - Unknown formats handled safely
#
# ### Setup:
# 1. Run all cells in order
# 2. Place your JSON export files in `/content/input_data/`
#    - ChatGPT: `conversations.json` (supports both mapping and messages format)
#    - Claude: `conversations.json` (rename to avoid conflicts)
#    - Gemini: any JSON export file
#    - JSONL: `.jsonl` files (per-conversation or per-message)
#
# ### Configuration:
# - Modify the CONFIG dictionary in the Configuration cell
# - Key options:
#   - `subset`: "both", "user", or "assistant"
#   - `format`: "md", "txt", or "json"
#   - `export_code`: True/False
#   - `keywords`: List of keywords for grouping
#   - `drive_out`: Path to copy outputs to Drive
#   - **v8 additions:**
#     - `build_sources`: Enable sources mode
#     - `sources_cap`: Maximum number of source documents (default: 150)
#     - `include_assistant_context`: Include assistant responses as blockquotes
#     - `min_user_chars`: Minimum characters for user text segments
#     - `code_only`: Export only code blocks
#     - `aggregate_keyword_files`: Create single file per keyword
#
# ### Output Structure:
# ```
# /content/chat_exports/
# ├── default/                    # Main transcripts
# ├── code_exports/               # Extracted code blocks
# ├── sources/                    # User text source documents (v8)
# ├── meta/                       # Metadata CSVs (v8)
# │   ├── sources_index.csv
# │   └── segments_index.csv
# └── keyword_{keyword}/          # Grouped by keywords
#     └── keyword_{keyword}.md    # Aggregated file (v8)
# ```
#
# ### File Naming:
# - Transcripts: `<title> - <YYYYMMDD_HHMMSS> - <subset>.<ext>`
# - Code: `<title> - code<nn>.<ext>`
# - Sources: `<canonical_title> - <YYYYMMDD_HHMMSS>.md`
# - Collisions handled with `-<hash>` suffix
#
# ### Features:
# - ✅ Handles UTF-16/32 with BOM
# - ✅ Detects and warns about ZIP/GZIP files
# - ✅ Streams large files (1GB+) with ijson
# - ✅ Supports JSONL format (per-conversation or per-message)
# - ✅ Preserves all platform parsing logic
# - ✅ Reconstructs ChatGPT thread order from mapping
# - ✅ Exports to flat, sortable structure
# - ✅ Optional Drive backup
# - ✅ Sources mode with provenance tracking
# - ✅ Keyword aggregation
# - ✅ Code-only export mode
# - ✅ Normalized timestamp handling across formats

✓ ijson installed (for large file support)
✓ Directories created
  Input:  /content/input_data/
  Output: /content/chat_exports/
Current Configuration:
----------------------------------------
input_root                  : /content/input_data
output_root                 : /content/chat_exports
subset                      : both
format                      : md
export_code                 : True
keywords                    : []
groups                      : non-unique
drive_out                   : None
max_filename_len            : 160
dry_run                     : False
verbose                     : True
build_sources               : True
sources_cap                 : 150
include_assistant_context   : False
min_user_chars              : 400
similarity_threshold        : 0.35
code_only                   : False
aggregate_keyword_files     : False
Found 4 JSON file(s) in /content/input_data
Processing: test_sources.json (0.0 MB)
  Platform: chatgpt, Conversations: 1
Processing: test_code

JSONDecodeError: Unexpected UTF-8 BOM (decode using utf-8-sig): line 1 column 1 (char 0)